# DGRL dg.o corpus: data enrichment and network pipeline (replication notebook)

This notebook reproduces every enrichment step behind the GIQ revision: DOIs, affiliations, abstracts, the master dataset, the topic sensitivity check, the co-authorship network, and the country and institution networks. Each value in the outputs records where it came from.

**How replication works**
- Every API response (OpenAlex, Crossref, ROR, Semantic Scholar) is saved to `api_cache.json` on first fetch. Rerunning with `REFRESH_API = False` reads the saved responses, so the results are identical even after the sources change. Set `REFRESH_API = True` only to deliberately pull fresh data.
- The DBLP records extracted from DBLP's official full download are saved to `dblp/dblp_dgo_records.json` and reused the same way.
- All human judgments are in the **Settings and manual decisions** cell, plus hand-collected logs (listed below).
- The last cell writes `run_manifest.json`: run date, package versions, checksums of every input and output, and the headline counts.

**Run order:** top to bottom, once. No cell edits files in place; each step writes its own output.

**Inputs (must exist on Drive before running)**

| File | Folder | Produced by |
|---|---|---|
| `dgrl_topics_k10.csv`, `dgrl_topics_k15.csv`, `dgrl_embeddings.npy` | `DGO_25_Years_Data/inputs/` (copied from `BERT/`) | `DGRL_Complete_Pipeline_Colab` (parsing, embeddings, BERTopic) |
| `openalex_matches_cache.json`, `openalex_authorships.csv` | `DGO_25_Years_Data/inputs/` | `DGRL_OpenAlex_v2_Colab` (verified OpenAlex matching; that notebook saved them as `openalex_v2_cache.json` and `openalex_v2_authorships.csv`, renamed here) |
| `manual_lookups_log.csv` | `DGO_25_Years_Data/inputs/` | Hand-collected abstracts, one row per paper with source URL |
| `affiliation_lookups_log.csv` | `DGO_25_Years_Data/inputs/` | Hand-collected affiliations, one row per author with source URL |
| `web_affiliation_lookups_2026-11-19.csv` | `DGO_25_Years_Data/inputs/` | Affiliations for the 118 papers still missing after Step 4, verified by hand from the paper, publisher records or repository copies; one row per author with source URL and evidence type |
| `coauthor_affiliation_lookups_2026-11-20.csv` | `DGO_25_Years_Data/inputs/` | Countries for co-authors that OpenAlex listed without an institution, from publisher records (Crossref) and the ACM Digital Library; one row per author with source URL |
| `paper_links_extra.csv` | `DGO_25_Years_Data/inputs/` | Web links (mostly DOIs) for papers whose DGRL record has no DOI, from exact title matches in OpenAlex; used for the dashboard's paper links |

Secrets: `OPENALEX_API_KEY` in Colab secrets. All other sources are free and keyless.

## Settings and manual decisions
Every human judgment used anywhere in the pipeline is defined here, with the reason.

In [ ]:
REFRESH_API = False     # False: reuse saved API responses (exact replication). True: fetch fresh data.

# Known false OpenAlex match: two different papers share this title and a surname (found during verification)
FALSE_MATCH_TITLES = ['teleworking in times of covid']

# DGRL title errors found during manual lookups (original titles are kept in title_original)
TITLE_FIX = {
    135: 'BPMapping and UMM in Business Process Modeling for e-Government Processes',   # DGRL typo "UPM"
    244: 'Laying the Foundations for Public Participation and Value Advocacy: Interaction Design for a Large Scale Urban Simulation',  # "Simluation"
    495: 'What should government agencies do with information that is relevant for citizens?',  # venue text appended in DGRL
    661: 'Data-centric workflows in government: a new avenue of research?',                    # venue text appended in DGRL
}

# Author IDs judged by hand to be the same person (name variants); all other merges are automatic and evidence-based
MANUAL_SAME_PAIRS = [('Gianluca Carlo Misuraca', 'Gianluca C. Misuraca'),
                     ('M. Ku / Minyoung Ku', 'Minyoung Ku'),
                     ('S.Y.W. Su', 'Stanley Y. W. Su'),
                     ('Sahng-Yoon Kim', 'Sahng Yoon Kim')]

# Historical phases (Reviewer 2's history of the society)
PHASES = [(2000, 2005, 'NSF PI meetings'), (2006, 2012, 'DGS of North America'), (2013, 2024, 'Digital Government Society')]

## Setup
Mounts Drive, installs packages, defines the cached API layer, and loads the inputs.

In [ ]:
!pip install -q rapidfuzz pycountry-convert sentence-transformers
import os, re, json, time, hashlib, urllib.parse, datetime
import pandas as pd, numpy as np, requests
from rapidfuzz import fuzz
from google.colab import drive, userdata
drive.mount('/content/drive')

REV = '/content/drive/MyDrive/DGO_25_Years_Data'   # project folder: every output is written here
IN = os.path.join(REV, 'inputs')                     # files this notebook reads but does not create
OUT = os.path.join(REV, 'gephi'); os.makedirs(OUT, exist_ok=True)
os.makedirs(os.path.join(REV, 'dblp'), exist_ok=True)
EMAIL = 'ChristopherReddish@USF.edu'
API_KEY = userdata.get('OPENALEX_API_KEY')
HEAD = {'User-Agent': f'DGRL-research (mailto:{EMAIL})'}
RUN_STARTED = datetime.datetime.now().isoformat(timespec='seconds')

# ---- cached API layer: every response is saved, so reruns reproduce the same data ----
API_CACHE_PATH = os.path.join(REV, 'api_cache.json')
API = json.load(open(API_CACHE_PATH)) if os.path.exists(API_CACHE_PATH) else {}
LAST_FROM_CACHE = False
def _api_key(url, params):
    p = {k: v for k, v in (params or {}).items() if k not in ('api_key', 'mailto')}
    return url + '?' + urllib.parse.urlencode(sorted(p.items()))
def get_json(url, params=None, tries=3, timeout=30, **_):
    global LAST_FROM_CACHE
    k = _api_key(url, params)
    if not REFRESH_API and k in API:
        LAST_FROM_CACHE = True
        return API[k]['data']
    LAST_FROM_CACHE = False
    for t in range(tries):
        try:
            r = requests.get(url, params=params, headers=HEAD, timeout=timeout)
            if r.status_code == 200:
                data = r.json(); API[k] = {'data': data, 'fetched': datetime.datetime.now().isoformat(timespec='seconds')}; return data
            if r.status_code == 404:
                API[k] = {'data': None, 'fetched': datetime.datetime.now().isoformat(timespec='seconds')}; return None
        except requests.RequestException:
            pass
        time.sleep(2 * (t + 1))
    return None                      # failures (e.g. rate limits) are not cached, so a rerun retries them
def pause(s):                        # be polite to live APIs; no waiting when the answer came from the cache
    if not LAST_FROM_CACHE: time.sleep(s)
def save_api_cache():
    json.dump(API, open(API_CACHE_PATH, 'w'))
    print(f"API cache saved: {len(API)} responses")
print(f"API cache: {len(API)} saved responses | REFRESH_API = {REFRESH_API}")

# ---- inputs ----
df = pd.read_csv(os.path.join(IN, 'dgrl_topics_k10.csv'))
cache = json.load(open(os.path.join(IN, 'openalex_matches_cache.json')))

FALSE_MATCHES = [i for i in range(len(df)) if any(t in str(df.loc[i, 'title']).lower() for t in FALSE_MATCH_TITLES)]
for i in FALSE_MATCHES:
    cache[str(i)] = {'status': 'no_match', 'note': 'false match removed'}

def norm(s):
    s = str(s).lower()
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def surname(authors):
    t = norm(str(authors).split(',')[0]).split()
    return t[-1] if t else ''

def has_country(rec):
    if not rec or rec.get('status') != 'matched':
        return False
    for a in rec.get('authorships', []):
        if a.get('countries') or any(ins.get('country') for ins in a.get('institutions', [])):
            return True
    return False

before = sum(has_country(cache.get(str(i))) for i in range(len(df)))
print(f"Papers: {len(df)} | with affiliation country before this pass: {before} ({before/len(df):.1%})")

## Step 1. DOIs
DOIs come from the verified OpenAlex match where available, otherwise from a Crossref title search (accepted only at title similarity 90+, or a partial match of 95+ with the first author's surname). DBLP's search API blocks automated requests, so DBLP is used later through its official download instead.

Adds `doi` and `doi_source` to the working table.

In [ ]:
def crossref_doi(row):
    t, y, ln = str(row['title']), int(row['year']), surname(row['authors'])
    js = get_json('https://api.crossref.org/works',
                  {'query.bibliographic': t, 'filter': f'from-pub-date:{y-1},until-pub-date:{y+1}',
                   'rows': 5, 'select': 'DOI,title,author', 'mailto': EMAIL}, tries=3, timeout=30)
    best = (0, None)
    for it in ((js or {}).get('message') or {}).get('items', []):
        ct = norm((it.get('title') or [''])[0])
        s = fuzz.token_sort_ratio(norm(t), ct)
        fams = ' '.join(norm(a.get('family', '')) for a in it.get('author', [])).split()
        if s < 90 and min(len(norm(t)), len(ct)) >= 25 and ln in fams:
            p = fuzz.partial_ratio(norm(t), ct)
            if p >= 95: s = p
        if s > best[0]:
            best = (s, it.get('DOI'))
    return best if best[0] >= 90 else (best[0], None)

dois, srcs = [], []
for i, row in df.iterrows():
    rec = cache.get(str(i), {})
    oa_doi = (rec.get('doi') or '').replace('https://doi.org/', '') if rec.get('status') == 'matched' else ''
    if oa_doi:
        dois.append(oa_doi.lower()); srcs.append('openalex')
    else:
        s, d = crossref_doi(row)
        dois.append(d.lower() if d else None); srcs.append('crossref' if d else None)
        pause(0.15)
    if (i + 1) % 100 == 0:
        print(f"[{i+1}/{len(df)}] DOIs so far: {sum(x is not None for x in dois)}")

df['doi'] = dois
df['doi_source'] = srcs
print(f"\nPapers with a DOI: {df.doi.notna().sum()}/{len(df)}")
print(df.doi_source.value_counts(dropna=False).to_string())
save_api_cache()

## Step 2. OpenAlex by DOI
Exact OpenAlex lookups by DOI for papers the title search missed (accepted at title similarity 85+). Single-record lookups are free.

In [ ]:
SELECT = 'id,doi,title,publication_year,authorships,cited_by_count,type,primary_location'
spent = {'usd': 0.0}
def oa_by_doi(doi):
    js = get_json(f'https://api.openalex.org/works/doi:{doi}', {'select': SELECT, 'api_key': API_KEY, 'mailto': EMAIL}, timeout=30)
    pause(0.12)
    return js

def to_record(w, strategy):
    auths = []
    for k, au in enumerate(w.get('authorships', [])):
        person = au.get('author') or {}
        insts = [{'id': x.get('id'), 'name': x.get('display_name'), 'country': x.get('country_code'), 'ror': x.get('ror')} for x in au.get('institutions', [])]
        auths.append({'order': k + 1, 'position': au.get('author_position'), 'author_id': person.get('id'),
                      'author_name': person.get('display_name'), 'countries': au.get('countries', []),
                      'institutions': insts, 'raw_affiliation': au.get('raw_affiliation_strings', [])})
    src = ((w.get('primary_location') or {}).get('source') or {})
    return {'status': 'matched', 'strategy': strategy, 'rule': 'doi_from_dblp',
            'score': None,
            'openalex_id': w.get('id'), 'doi': w.get('doi'), 'oa_title': w.get('title'), 'oa_year': w.get('publication_year'),
            'type': w.get('type'), 'venue': src.get('display_name'), 'cited_by_count': w.get('cited_by_count') or 0, 'authorships': auths}

targets = [i for i in range(len(df)) if cache.get(str(i), {}).get('status') != 'matched' and isinstance(df.loc[i, 'doi'], str)]
print(f"Unmatched papers that now have a DOI: {len(targets)}")
new_oa = 0
for i in targets:
    w = oa_by_doi(df.loc[i, 'doi'])
    if not w:
        continue
    sim = fuzz.token_sort_ratio(norm(df.loc[i, 'title']), norm(w.get('title')))
    if sim >= 85:
        rec = to_record(w, 'openalex_by_doi')
        rec['score'] = sim
        cache[str(i)] = rec
        new_oa += 1
print(f"New OpenAlex matches by DOI: {new_oa}")
save_api_cache()

## Step 3. Affiliations from ROR and Crossref
Affiliation text (OpenAlex raw strings, then Crossref author affiliations) is matched to an institution and country with the Research Organization Registry (ROR), using only ROR's own chosen match.

In [ ]:
ROR_CACHE_PATH = os.path.join(REV, 'ror_affiliation_cache.json')
ror_cache = json.load(open(ROR_CACHE_PATH)) if os.path.exists(ROR_CACHE_PATH) else {}

def ror_match(aff):
    aff = re.sub(r'\s+', ' ', str(aff)).strip()
    if len(aff) < 4:
        return None
    if aff in ror_cache:
        return ror_cache[aff]
    js = get_json('https://api.ror.org/v2/organizations', {'affiliation': aff}, timeout=30)
    out = None
    for it in (js or {}).get('items', []):
        if it.get('chosen'):
            org = it.get('organization', {})
            name = next((n.get('value') for n in org.get('names', []) if 'ror_display' in (n.get('types') or [])), None)
            locs = org.get('locations') or [{}]
            cc = ((locs[0] or {}).get('geonames_details') or {}).get('country_code')
            out = {'ror': org.get('id'), 'name': name, 'country': cc}
            break
    ror_cache[aff] = out
    pause(0.2)
    return out

print("ROR matcher ready. Cached strings:", len(ror_cache))

fill = {}   # row -> {'source', 'authors': [{'order', 'name', 'institutions': [{name, country, ror}]}]}

# 6a. OpenAlex records that have raw affiliation text but no structured institution or country
for i in range(len(df)):
    rec = cache.get(str(i), {})
    if rec.get('status') != 'matched' or has_country(rec):
        continue
    auths = []
    for a in rec.get('authorships', []):
        insts = [m for m in (ror_match(s) for s in a.get('raw_affiliation', [])) if m and m.get('country')]
        auths.append({'order': a['order'], 'name': a.get('author_name'), 'institutions': insts})
    if any(a['institutions'] for a in auths):
        fill[i] = {'source': 'openalex_raw_affiliation+ror', 'authors': auths}
print(f"Filled from OpenAlex raw affiliation strings: {len(fill)}")

# 6b. Crossref affiliation strings for anything still without a country but with a DOI
n_cr = 0
for i in range(len(df)):
    if i in fill or has_country(cache.get(str(i))):
        continue
    doi = df.loc[i, 'doi'] if isinstance(df.loc[i, 'doi'], str) else (cache.get(str(i), {}).get('doi') or '').replace('https://doi.org/', '')
    if not doi:
        continue
    js = get_json(f'https://api.crossref.org/works/{urllib.parse.quote(doi)}', {'mailto': EMAIL}, timeout=30)
    pause(0.2)
    authors = ((js or {}).get('message') or {}).get('author', []) or []
    auths = []
    for k, a in enumerate(authors):
        insts = [m for m in (ror_match(x.get('name', '')) for x in a.get('affiliation', [])) if m and m.get('country')]
        auths.append({'order': k + 1, 'name': f"{a.get('given', '')} {a.get('family', '')}".strip(), 'institutions': insts})
    if any(a['institutions'] for a in auths):
        fill[i] = {'source': 'crossref+ror', 'authors': auths}
        n_cr += 1
print(f"Filled from Crossref: {n_cr}")
json.dump(ror_cache, open(ROR_CACHE_PATH, 'w'))
save_api_cache()

## Step 4. Combine affiliation sources
One row per paper with all authors' countries, the first author's countries, regions, institutions, and `affiliation_source`.

Writes `affiliations.csv` and `still_missing_for_manual_entry.csv`.

In [ ]:
import pycountry_convert as pc
def region(cc):
    if cc in ('US', 'CA'): return 'North America'
    try:
        return {'NA': 'Latin America', 'SA': 'Latin America', 'EU': 'Europe', 'AS': 'Asia', 'AF': 'Africa', 'OC': 'Oceania'}[pc.country_alpha2_to_continent_code(cc)]
    except Exception:
        return ''

out = []
for i, row in df.iterrows():
    rec = cache.get(str(i), {})
    if has_country(rec):
        auths = [{'order': a['order'], 'name': a.get('author_name'),
                  'countries': set(a.get('countries') or []) | {x['country'] for x in a['institutions'] if x.get('country')},
                  'insts': [x['name'] for x in a['institutions'] if x.get('name')]} for a in rec['authorships']]
        src = 'openalex_by_doi' if rec.get('strategy') == 'openalex_by_doi' else 'openalex'
    elif i in fill:
        auths = [{'order': a['order'], 'name': a['name'], 'countries': {x['country'] for x in a['institutions']},
                  'insts': [x['name'] for x in a['institutions'] if x.get('name')]} for a in fill[i]['authors']]
        src = fill[i]['source']
    else:
        auths, src = [], 'missing'
    allc = sorted(set().union(*[a['countries'] for a in auths])) if auths else []
    first = sorted(auths[0]['countries']) if auths else []
    out.append({'row_id': i, 'year': int(row['year']), 'topic': int(row['topic']), 'title': row['title'], 'authors_dgrl': row['authors'],
                'doi': row['doi'], 'affiliation_source': src,
                'countries_all': '|'.join(allc), 'countries_first_author': '|'.join(first), 'n_countries': len(allc),
                'regions_all': '|'.join(sorted({region(c) for c in allc} - {''})),
                'institutions_all': '|'.join(sorted({n for a in auths for n in a['insts']}))})
aff_out = pd.DataFrame(out)
aff_out['period'] = pd.cut(aff_out.year, [1999, 2004, 2009, 2014, 2019, 2024], labels=['2000-04', '2005-09', '2010-14', '2015-19', '2020-24'])
aff_out.drop(columns=['period']).to_csv(os.path.join(REV, 'affiliations.csv'), index=False)

after = (aff_out.n_countries > 0).sum()
print(f"Affiliation country coverage: {before} ({before/len(df):.1%}) -> {after} ({after/len(df):.1%})")
print("\nBy source:\n", aff_out.affiliation_source.value_counts().to_string())
print("\nCoverage by period:")
print(aff_out.groupby('period', observed=True).apply(lambda g: f"{(g.n_countries > 0).mean():.0%} of {len(g)}").to_string())

left = df.loc[aff_out.n_countries.eq(0).values, ['title', 'year', 'authors', 'citation', 'doi']].copy()
for c in ['first_author_institution_manual', 'first_author_country_manual', 'all_author_countries_manual', 'notes']:
    left[c] = ''
left.sort_values('year').to_csv(os.path.join(REV, 'still_missing_for_manual_entry.csv'))
print(f"\nStill missing, saved for manual entry: {len(left)}")

## Step 5. Automated abstract backfill
For papers without an abstract: Crossref, then OpenAlex by DOI, then Semantic Scholar by DOI, then Semantic Scholar by title (accepted only at title similarity 85+).

Writes `abstracts_backfill.csv` and `still_missing_abstracts.csv`. Abstracts found by hand afterwards are in `manual_lookups_log.csv`.

In [ ]:
def has_abs(x):
    return isinstance(x, str) and len(x.strip()) > 10

def clean_abs(t):
    t = re.sub(r'<[^>]+>', ' ', str(t or ''))
    t = re.sub(r'^\s*abstract\s*', '', t, flags=re.I)
    return re.sub(r'\s+', ' ', t).strip()

def from_inverted(inv):
    if not inv: return ''
    pos = sorted((p, w) for w, ps in inv.items() for p in ps)
    return ' '.join(w for _, w in pos)

def title_ok(found_title, i):
    return fuzz.token_sort_ratio(norm(found_title), norm(df.loc[i, 'title'])) >= 85

todo = [i for i in range(len(df)) if not has_abs(df.loc[i, 'abstract'])]
print(f"Missing abstracts: {len(todo)}")
found = {}
for i in todo:
    doi = df.loc[i, 'doi'] if isinstance(df.loc[i, 'doi'], str) else None
    abs_txt, src = '', None
    if doi:
        js = get_json(f'https://api.crossref.org/works/{urllib.parse.quote(doi)}', {'mailto': EMAIL}, timeout=30)
        m = (js or {}).get('message') or {}
        if m.get('abstract'):
            abs_txt, src = clean_abs(m['abstract']), 'crossref'
        if len(abs_txt) < 100:
            w = get_json(f'https://api.openalex.org/works/doi:{doi}', {'select': 'title,abstract_inverted_index', 'api_key': API_KEY, 'mailto': EMAIL}, timeout=30)
            t = from_inverted((w or {}).get('abstract_inverted_index'))
            if len(t) >= 100:
                abs_txt, src = t, 'openalex_doi'
        if len(abs_txt) < 100:
            s2 = get_json(f'https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}', {'fields': 'title,abstract'}, timeout=30)
            if s2 and s2.get('abstract'):
                abs_txt, src = clean_abs(s2['abstract']), 'semantic_scholar_doi'
            pause(1.2)
    if len(abs_txt) < 100:
        s2 = get_json('https://api.semanticscholar.org/graph/v1/paper/search/match', {'query': df.loc[i, 'title'], 'fields': 'title,year,abstract'}, timeout=30)
        hit = ((s2 or {}).get('data') or [{}])[0]
        if hit.get('abstract') and title_ok(hit.get('title', ''), i):
            abs_txt, src = clean_abs(hit['abstract']), 'semantic_scholar_title'
        pause(1.2)
    if len(abs_txt) >= 100:
        found[i] = (abs_txt, src)

ab = pd.DataFrame([{'row_id': i, 'year': int(df.loc[i, 'year']), 'title': df.loc[i, 'title'], 'doi': df.loc[i, 'doi'],
                    'abstract_source': found[i][1] if i in found else 'missing',
                    'abstract': found[i][0] if i in found else ''} for i in todo])
ab.to_csv(os.path.join(REV, 'abstracts_backfill.csv'), index=False)
left = ab[ab.abstract_source == 'missing'].copy()
left['acm_link'] = left.doi.map(lambda d: f'https://doi.org/{d}' if isinstance(d, str) else '')
left['abstract_manual'] = ''
left.drop(columns=['abstract', 'abstract_source']).to_csv(os.path.join(REV, 'still_missing_abstracts.csv'), index=False)

print(f"Recovered: {len(found)} of {len(todo)}")
print(ab.abstract_source.value_counts().to_string())
print("\nStill missing by year:\n", left.groupby('year').size().to_string())
save_api_cache()

## Step 5b. Combine the affiliation logs
Merges the original hand-collected log with the web lookups for the 118 papers that had no affiliation after Step 4. Every web lookup row carries its source URL and evidence type; 3 papers whose institution comes from a related page rather than the paper itself can be left out with `INCLUDE_WEAKER = False`. Inputs are not edited.

Writes `affiliation_lookups_combined.csv`, which Steps 6, 9 and 12b read.


In [ ]:
# Combines the two hand-collected affiliation logs into one file that Steps 6, 9 and 12b read.
# Inputs are not edited: the combined log is written to affiliation_lookups_combined.csv.
import os, pandas as pd

INCLUDE_WEAKER = True   # False leaves out the 3 papers whose institution comes from a related page, not the paper (rows 497, 680, 804)

log = pd.read_csv(os.path.join(IN, 'affiliation_lookups_log.csv'), dtype={'row_id': int})
web = pd.read_csv(os.path.join(IN, 'web_affiliation_lookups_2026-11-19.csv'), dtype={'row_id': int}).fillna('')
weaker = set(web.loc[web.notes.str.contains('Weaker evidence', case=False), 'row_id'])
if not INCLUDE_WEAKER:
    web = web[~web.row_id.isin(weaker)]
web = web[~web.row_id.isin(set(log.row_id))]          # the original log wins if a paper is in both
co_path = os.path.join(IN, 'coauthor_affiliation_lookups_2026-11-20.csv')   # co-authors with no country on papers that had one
if not os.path.exists(co_path):
    raise FileNotFoundError(f"{co_path} not found. If it was just replaced on Drive, Drive may still be syncing: Runtime > Disconnect and delete runtime, then Run all.")
co = pd.read_csv(co_path, dtype={'row_id': int}).fillna('')
seen = set(zip(log.row_id, log.author)) | set(zip(web.row_id, web.author))
co = co[pd.Series([(r, a) not in seen for r, a in zip(co.row_id, co.author)], index=co.index, dtype=bool)]
web = pd.concat([web, co[log.columns]], ignore_index=True)
AFF_LOG = os.path.join(REV, 'affiliation_lookups_combined.csv')
ORIG_LOG_IDS = set(log.row_id)   # Step 9 uses these author lists first; web lookup lists only when nothing else lists the paper
comb = pd.concat([log, web[log.columns]], ignore_index=True).sort_values(['row_id', 'author_order'], kind='stable')
comb.to_csv(AFF_LOG, index=False)
print(f"Co-author lookups: {len(co)} authors on {co.row_id.nunique()} papers")
print(f"Affiliation log: {log.row_id.nunique()} papers (original) + {web.row_id.nunique()} papers (web and co-author lookups) = "
      f"{comb.row_id.nunique()} papers, {len(comb)} author rows | weaker-evidence papers "
      f"{'included' if INCLUDE_WEAKER else 'left out'}: {sorted(weaker)}")


## Step 6. Master dataset
Combines the original corpus, automated and manual abstracts, title corrections, and affiliations, then applies the manual affiliation log (only where a paper had no country data) and joins both topic models.

Provenance columns: `abstract_source`, `abstract_source_url`, `affiliation_source`, `title_original`, `abstract_original`, `title_corrected`, `abstract_added_after_model`.

Writes `dgrl_master_k10.csv`, `dgrl_master_k15.csv`, and the combined `dgrl_master.csv`. Topic assignments are not changed here.

In [ ]:
import os, pandas as pd

manual_path = os.path.join(IN, 'manual_lookups_log.csv')
if os.path.exists(manual_path):
    manual = pd.read_csv(manual_path, dtype=str).fillna('')
else:
    print(f"Warning: {manual_path} not found. Proceeding with an empty manual log.")
    manual = pd.DataFrame(columns=['row_id', 'field', 'value', 'source_type', 'source_url'])

bf = pd.read_csv(os.path.join(REV, 'abstracts_backfill.csv'), dtype=str).fillna('')
aff = pd.read_csv(os.path.join(REV, 'affiliations.csv'))

# Title errors found in DGRL during lookups (originals kept in title_original)

def has_text(x):
    return isinstance(x, str) and len(x.strip()) > 10

auto = {int(r.row_id): (r.abstract, f'automated: {r.abstract_source}', '')
        for r in bf.itertuples() if r.abstract_source != 'missing' and has_text(r.abstract)}
man = {int(r.row_id): (r.value, f'manual: {r.source_type}', r.source_url)
       for r in manual.itertuples() if r.field == 'abstract' and has_text(r.value)}

aff_cols = ['row_id', 'doi', 'affiliation_source', 'countries_all', 'countries_first_author',
            'n_countries', 'regions_all', 'institutions_all']

for k in (10, 15):
    m = pd.read_csv(os.path.join(IN, f'dgrl_topics_k{k}.csv'))
    m = m.drop(columns=['countries', 'country_count'], errors='ignore')
    m.insert(0, 'row_id', range(len(m)))

    m['title_original'] = m['title']
    m['title'] = [TITLE_FIX.get(i, t) for i, t in zip(m.row_id, m.title)]
    m['title_corrected'] = m.row_id.isin(list(TITLE_FIX))

    finals, sources, urls = [], [], []
    for i, a in zip(m.row_id, m.abstract):
        if has_text(a):
            finals.append(a); sources.append('original corpus (DGRL + March 2026 OpenAlex backfill)'); urls.append('')
        elif i in auto:
            finals.append(auto[i][0]); sources.append(auto[i][1]); urls.append(auto[i][2])
        elif i in man:
            finals.append(man[i][0]); sources.append(man[i][1]); urls.append(man[i][2])
        else:
            finals.append(''); sources.append('none (title only)'); urls.append('')
    m['abstract_original'] = m['abstract']
    m['abstract'] = finals
    m['abstract_source'] = sources
    m['abstract_source_url'] = urls
    m['abstract_added_after_model'] = ~m.abstract_original.apply(has_text) & (m.abstract != '')
    m['combined_text_updated'] = (m.title + '. ' + m.abstract).str.strip()

    m = m.merge(aff[aff_cols], on='row_id', how='left')
    out = os.path.join(REV, f'dgrl_master_k{k}.csv')
    m.to_csv(out, index=False)
    print(f"Saved {os.path.basename(out)}: {m.shape}")

print(f"\nAbstracts: {(m.abstract != '').sum()} of {len(m)}")
print(m.abstract_source.value_counts().to_string())
print(f"\nAdded after the topic model was fit (need refit or reassign): {m.abstract_added_after_model.sum()}")
print(f"Titles corrected: {m.title_corrected.sum()}")
print(f"Affiliation country coverage: {(m.n_countries.fillna(0) > 0).sum()} of {len(m)}")

In [ ]:
import os, pandas as pd, pycountry_convert as pc
log = pd.read_csv(AFF_LOG)

def region(cc):
    if cc in ('US', 'CA'): return 'North America'
    try:
        return {'NA': 'Latin America', 'SA': 'Latin America', 'EU': 'Europe', 'AS': 'Asia',
                'AF': 'Africa', 'OC': 'Oceania'}[pc.country_alpha2_to_continent_code(cc)]
    except Exception:
        return ''

fill = {}
for rid, g in log.sort_values(['row_id', 'author_order']).groupby('row_id'):
    cs = sorted(set(g.country))
    fill[rid] = {'countries_all': '|'.join(cs),
                 'countries_first_author': g.iloc[0].country,
                 'n_countries': len(cs),
                 'regions_all': '|'.join(sorted({region(c) for c in cs} - {''})),
                 'institutions_all': '|'.join(sorted(set(g.institution))),
                 'affiliation_source': 'manual: ' + g.iloc[0].source_type}

for k in (10, 15):
    path = os.path.join(REV, f'dgrl_master_k{k}.csv')
    m = pd.read_csv(path)
    before = (m.n_countries.fillna(0) > 0).sum()
    applied = 0
    for rid, vals in fill.items():
        i = m.index[m.row_id == rid]
        if len(i) and not (m.loc[i[0], 'n_countries'] > 0):
            for c, v in vals.items():
                m.loc[i[0], c] = v
            applied += 1
    # papers that already had countries: add the countries of co-authors found by hand (paper level counts every author)
    added = 0
    for rid, g in log.groupby('row_id'):
        i = m.index[m.row_id == rid]
        if not len(i) or not (m.loc[i[0], 'n_countries'] > 0): continue
        cur = set(str(m.loc[i[0], 'countries_all']).split('|')) - {'', 'nan'}
        new = cur | set(g.country)
        if new != cur:
            m.loc[i[0], 'countries_all'] = '|'.join(sorted(new)); m.loc[i[0], 'n_countries'] = len(new)
            m.loc[i[0], 'regions_all'] = '|'.join(sorted({region(c) for c in new} - {''}))
            inst = set(str(m.loc[i[0], 'institutions_all']).split('|')) - {'', 'nan'}
            m.loc[i[0], 'institutions_all'] = '|'.join(sorted(inst | set(g.institution) - {''}))
            added += 1
    print(f"k={k}: added co-author countries to {added} papers that already had a country")
    m.to_csv(path, index=False)
    after = (m.n_countries.fillna(0) > 0).sum()
    print(f"k={k}: applied {applied} | affiliation coverage {before} -> {after} of {len(m)} ({after/len(m):.1%})")

In [ ]:
import os, pandas as pd
m10 = pd.read_csv(os.path.join(REV, 'dgrl_master_k10.csv'))
m15 = pd.read_csv(os.path.join(REV, 'dgrl_master_k15.csv'))

TABLE2 = {0: "Strategy, governance and collaboration", 1: "Open government and open data",
          2: "Digital platforms and digital transformation", 3: "Data Science and predictive analytics in government",
          4: "Social media in government", 5: "e-Participation", 6: "Smart cities", 7: "Emergency management",
          8: "AI and algorithmic policy making", 9: "Blockchain and innovation"}

m10 = m10.rename(columns={'topic': 'topic_k10', 'topic_label': 'topic_keywords_k10'})
m10.insert(m10.columns.get_loc('topic_k10') + 1, 'topic_name_k10', m10.topic_k10.map(TABLE2))
k15 = m15[['row_id', 'topic', 'topic_label']].rename(columns={'topic': 'topic_k15', 'topic_label': 'topic_keywords_k15'})
master = m10.merge(k15, on='row_id', how='left')

out = os.path.join(REV, 'dgrl_master.csv')
master.to_csv(out, index=False)
print(f"Saved {os.path.basename(out)}: {master.shape}")
print(master[['row_id', 'topic_k10', 'topic_name_k10', 'topic_k15']].head())

## Step 7. Topic sensitivity check
The 41 papers whose text changed after the model was fit are embedded with the same model (all-MiniLM-L6-v2) and assigned to the nearest existing topic centroid. Adds `topic_k10_reassigned`, `topic_k10_moved`, `topic_k15_reassigned`, `topic_k15_moved`; original topic columns are unchanged.

In [ ]:
import numpy as np, os, pandas as pd
from sentence_transformers import SentenceTransformer
master = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
emb = np.load(os.path.join(IN, 'dgrl_embeddings.npy'))        # original embeddings, same row order
changed = (master.abstract_added_after_model | master.title_corrected).values
print(f"Papers with new text: {changed.sum()}")

model = SentenceTransformer('all-MiniLM-L6-v2')                        # same embedding model as the original run
new_emb = model.encode(master.loc[changed, 'combined_text_updated'].tolist(), normalize_embeddings=True)
unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)

for k in (10, 15):
    col = f'topic_k{k}'
    topics = master[col].values
    # centroids from papers whose text did NOT change, so title-only embeddings don't distort them
    cents = np.vstack([unit[(topics == t) & ~changed].mean(0) for t in range(k)])
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    reassigned = (new_emb @ cents.T).argmax(1)
    master[f'{col}_reassigned'] = topics
    master.loc[changed, f'{col}_reassigned'] = reassigned
    master[f'{col}_moved'] = master[col] != master[f'{col}_reassigned']
    moved = master.loc[changed, f'{col}_moved']
    print(f"\nk={k}: {moved.sum()} of {changed.sum()} updated papers would change topic")
    print(master.loc[changed & master[f'{col}_moved'], ['row_id', 'year', col, f'{col}_reassigned']].to_string(index=False))

master.to_csv(os.path.join(REV, 'dgrl_master.csv'), index=False)
print("\nSaved: original topic columns unchanged; new columns topic_k10_reassigned, topic_k10_moved, topic_k15_reassigned, topic_k15_moved")

## Step 8. Key numbers by historical phase
Papers, coverage, countries, international co-authorship, regional shares (fractional counting), and topic shares per phase. Writes `key_numbers_by_phase.csv`.

In [ ]:
import os, pandas as pd
m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))


def phase(y):
    for a, b, name in PHASES:
        if a <= y <= b: return f'{a}-{b} {name}'
m['phase'] = m.year.map(phase)
order = [f'{a}-{b} {n}' for a, b, n in PHASES]
g = m[m.n_countries.fillna(0) > 0]

rows = []
for p in order:
    s, sg = m[m.phase == p], g[g.phase == p]
    ccs = sg.countries_all.str.split('|').explode()
    regs = sg.regions_all.fillna('').str.split('|')
    w = pd.Series([r for rs in regs for r in rs for _ in [1]],
                  dtype=object).value_counts() if len(sg) else pd.Series(dtype=float)
    frac = {}
    for rs in regs:
        rs = [r for r in rs if r]
        for r in rs: frac[r] = frac.get(r, 0) + 1 / len(rs)
    tot = sum(frac.values()) or 1
    rows.append({'phase': p, 'papers': len(s), 'with_country': len(sg),
                 'coverage_%': round(len(sg) / len(s) * 100, 1), 'countries': ccs.nunique(),
                 'intl_coauthored_%': round((sg.n_countries > 1).mean() * 100, 1),
                 **{f'{r}_%': round(frac.get(r, 0) / tot * 100, 1) for r in
                    ['North America', 'Europe', 'Latin America', 'Asia', 'Africa', 'Oceania']}})
out = pd.DataFrame(rows)
print(out.to_string(index=False))
topics = pd.crosstab(m.phase, m.topic_name_k10, normalize='index').reindex(order).mul(100).round(1)
print("\nTopic share by phase (%):\n", topics.T.to_string())
out.to_csv(os.path.join(REV, 'key_numbers_by_phase.csv'), index=False)

## Step 9. Author lists, stage 1
Starts from OpenAlex authorships (unique author IDs), then adds, in order: OpenAlex by DOI, the manual affiliation log, Crossref author lists, and DGRL citations that list every author (no "et al."). Name-only authors are linked to an existing OpenAlex author only when exactly one candidate shares the surname and first initial.

Writes `authorships_stage1.csv` (one row per author per paper, with `author_source`).

The original hand-collected affiliation log is used as an author list before Crossref; the web lookups (some list only the verified authors) are used in Step 11, after DBLP, only for papers no other source lists.


In [ ]:
import os, re, json, time, itertools, unicodedata, requests, pandas as pd, networkx as nx
from rapidfuzz import fuzz

m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
oa = pd.read_csv(os.path.join(IN, 'openalex_authorships.csv'))
mlog = pd.read_csv(AFF_LOG)
false_match = set(m.loc[m.title.str.contains('Teleworking in times of COVID', case=False, na=False), 'row_id'])
oa = oa[~oa.row_id.isin(false_match) & oa.author_id.notna()]

def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z ]+', ' ', s)).strip()
def key_from(first, last):
    f, l = fold(first), fold(last).replace(' ', '')
    return f'{l}_{f[:1]}' if l else ''
def key_full(name):
    t = fold(name).split()
    return key_from(t[0], t[-1]) if len(t) >= 2 else ''
rows = [dict(row_id=r.row_id, order=r.order, author_key=r.author_id.replace('https://openalex.org/', ''),
             author_name=r.author_name, institution=r.institution, country=r.country, author_source='openalex')
        for r in oa.itertuples()]
have = set(oa.row_id)

# name index of known OpenAlex authors, for matching name-only sources
idx = {}
for r in oa.drop_duplicates('author_id').itertuples():
    idx.setdefault(key_full(r.author_name), set()).add(r.author_id.replace('https://openalex.org/', ''))
def resolve(name, first=None, last=None):
    k = key_from(first, last) if last else key_full(name)
    c = idx.get(k, set())
    return (next(iter(c)), 'name_matched_to_openalex') if len(c) == 1 else (f'name:{k}', 'name_only')

todo = m[~m.row_id.isin(have)]
print(f"Papers not yet in the network: {len(todo)}")
stats = {}

# 1) OpenAlex by DOI
for r in todo.itertuples():
    doi = r.doi if isinstance(r.doi, str) and r.doi else None
    if not doi: continue
    w = get_json(f'https://api.openalex.org/works/doi:{doi}', {'select': 'id,title,authorships', 'api_key': API_KEY, 'mailto': EMAIL})
    pause(0.12)
    if not w or fuzz.token_sort_ratio(fold(w.get('title')), fold(r.title)) < 85: continue
    auths = [a for a in w.get('authorships', []) if (a.get('author') or {}).get('id')]
    if not auths: continue
    for k, a in enumerate(auths, 1):
        inst = (a.get('institutions') or [{}])
        rows.append(dict(row_id=r.row_id, order=k, author_key=a['author']['id'].replace('https://openalex.org/', ''),
                         author_name=a['author'].get('display_name'), institution=(inst[0] or {}).get('display_name') if inst else None,
                         country=(inst[0] or {}).get('country_code') if inst else None, author_source='openalex_by_doi'))
    have.add(r.row_id); stats['openalex_by_doi'] = stats.get('openalex_by_doi', 0) + 1

# 2) original manual affiliation log (full author lists, hand-collected before the web lookups)
for rid, g in mlog[mlog.row_id.isin(ORIG_LOG_IDS)].groupby('row_id'):
    if rid in have: continue
    for a in g.sort_values('author_order').itertuples():
        key, how = resolve(a.author)
        rows.append(dict(row_id=rid, order=a.author_order, author_key=key, author_name=a.author, institution=a.institution,
                         country=a.country, author_source=f'manual_log ({how})'))
    have.add(rid); stats['manual_log'] = stats.get('manual_log', 0) + 1

# 3) Crossref author lists
for r in m[~m.row_id.isin(have)].itertuples():
    doi = r.doi if isinstance(r.doi, str) and r.doi else None
    if not doi: continue
    js = get_json(f'https://api.crossref.org/works/{requests.utils.quote(doi)}', {'mailto': EMAIL})
    pause(0.2)
    msg = (js or {}).get('message') or {}
    if fuzz.token_sort_ratio(fold((msg.get('title') or [''])[0]), fold(r.title)) < 85: continue
    au = [a for a in msg.get('author', []) if a.get('family')]
    if not au: continue
    for k, a in enumerate(au, 1):
        nm = f"{a.get('given', '')} {a['family']}".strip()
        key, how = resolve(nm, a.get('given', ''), a['family'])
        rows.append(dict(row_id=r.row_id, order=k, author_key=key, author_name=nm, institution=None, country=None,
                         author_source=f'crossref ({how})'))
    have.add(r.row_id); stats['crossref'] = stats.get('crossref', 0) + 1

# 4) DGRL citations, only complete author lists (no "et al.")
def parse_dgrl(s):
    s = str(s).strip().rstrip('.')
    if not s or 'et al' in s.lower(): return []
    parts = [p.strip() for p in re.split(r',?\s+and\s+|\s*&\s*', s) if p.strip()]
    out = []
    head = parts[0]
    m2 = re.findall(r"([^,]+),\s*((?:[A-Z][a-z]*\.?\s*-?)+)", head)
    for last, ini in m2: out.append((ini.strip(), last.strip()))
    for p in parts[1:]:
        t = p.split()
        if len(t) >= 2: out.append((' '.join(t[:-1]), t[-1]))
    return out
for r in m[~m.row_id.isin(have)].itertuples():
    ppl = parse_dgrl(r.authors)
    if not ppl: continue
    for k, (first, last) in enumerate(ppl, 1):
        key, how = resolve(f'{first} {last}', first, last)
        rows.append(dict(row_id=r.row_id, order=k, author_key=key, author_name=f'{first} {last}', institution=None,
                         country=None, author_source=f'dgrl_citation ({how})'))
    have.add(r.row_id); stats['dgrl_citation'] = stats.get('dgrl_citation', 0) + 1

A = pd.DataFrame(rows).drop_duplicates(['row_id', 'author_key'])
A = A.merge(m[['row_id', 'year', 'topic_name_k10']], on='row_id', how='left')
A.to_csv(os.path.join(REV, 'authorships_stage1.csv'), index=False)
print("Papers added by source:", stats)
print(f"Papers in network now: {A.row_id.nunique()} of {len(m)} ({A.row_id.nunique()/len(m):.1%})")
print("Author rows by source:\n", A.author_source.value_counts().to_string())
save_api_cache()

## Step 10. DBLP records
DBLP's official full-database download is the sanctioned route for bulk use. Every dg.o paper is extracted (each record parsed as XML, so multi-line titles are captured) and saved to `dblp/dblp_dgo_records.json` with the download date and file checksum. Reruns reuse that file.

In [ ]:
import gzip, html
from lxml import etree
DBLP_RECORDS = os.path.join(REV, 'dblp', 'dblp_dgo_records.json')
if os.path.exists(DBLP_RECORDS) and not REFRESH_API:
    meta = json.load(open(DBLP_RECORDS)); recs = meta['records']
    print(f"Reusing saved DBLP extract from {meta['downloaded']} ({len(recs)} records, dump md5 {meta['dump_md5']})")
else:
    gz = '/content/dblp.xml.gz'
    if not os.path.exists(gz):
        for url in ['https://dblp.org/xml/dblp.xml.gz', 'https://dblp.uni-trier.de/xml/dblp.xml.gz']:
            try:
                print("Downloading", url)
                with requests.get(url, stream=True, timeout=60, headers=HEAD) as r:
                    r.raise_for_status()
                    with open(gz, 'wb') as f:
                        for i, ch in enumerate(r.iter_content(1 << 20)):
                            f.write(ch)
                            if i % 300 == 0: print(f"  {i} MB")
                if open(gz, 'rb').read(2) == b'\x1f\x8b': break
                os.remove(gz)
            except Exception as e:
                print("  failed:", str(e)[:80])
                if os.path.exists(gz): os.remove(gz)
    if not os.path.exists(gz): raise SystemExit("DBLP download blocked from Colab.")
    XML_ENT = {'amp', 'lt', 'gt', 'quot', 'apos'}
    def fix_entities(s):   # turn &uuml; etc. into real characters, keep XML's own five
        return re.sub(r'&([A-Za-z][A-Za-z0-9]*);', lambda mm: mm.group(0) if mm.group(1) in XML_ENT else html.unescape(mm.group(0)), s)
    def text_of(el):
        return re.sub(r'\s+', ' ', ''.join(el.itertext())).strip() if el is not None else ''

    print("Reading DBLP (several minutes)...")
    t0, recs, bad, buf, inrec = time.time(), [], [], [], False
    start = re.compile(r'<inproceedings\b[^>]*\bkey="conf/dgo/')
    with gzip.open(gz, 'rt', encoding='latin-1') as f:
        for line in f:
            if not inrec:
                mt = start.search(line)
                if not mt: continue
                inrec, buf = True, [line[mt.start():]]
            else:
                buf.append(line)
            if '</inproceedings>' in buf[-1]:
                block = ''.join(buf); block = block[:block.index('</inproceedings>') + len('</inproceedings>')]
                inrec, buf = False, []
                try:
                    el = etree.fromstring(fix_entities(block).encode('utf-8'), etree.XMLParser(recover=True))
                except Exception:
                    bad.append(block[:300]); continue
                recs.append(dict(key=el.get('key'), title=text_of(el.find('title')),
                                 year=int(el.findtext('year') or 0) if (el.findtext('year') or '').strip().isdigit() else 0,
                                 authors=[re.sub(r'\s+\d{4}$', '', text_of(a)) for a in el.findall('author')],
                                 ee=el.findtext('ee') or '', raw=block[:400]))
    if not recs: raise SystemExit(f"No dg.o records parsed. Unparseable blocks: {len(bad)}")
    md5 = hashlib.md5()
    with open(gz, 'rb') as fh:
        for ch in iter(lambda: fh.read(1 << 20), b''): md5.update(ch)
    json.dump({'downloaded': datetime.datetime.now().isoformat(timespec='seconds'), 'source': 'https://dblp.org/xml/dblp.xml.gz',
               'dump_md5': md5.hexdigest(), 'records': recs}, open(DBLP_RECORDS, 'w'))
    print(f"Saved {len(recs)} dg.o records to {DBLP_RECORDS}")
D = pd.DataFrame(recs)
D = D[D.authors.map(len) > 0].copy()
print(f"dg.o papers in DBLP: {len(D)} | empty titles: {(D.title == '').sum()}")

## Step 11. DBLP authors, duplicate resolution, co-authorship network
Missing papers are matched to DBLP by title within one year (90+, or partial 95+ with the first author's surname); DBLP author numbers like "0001" are removed. Author IDs sharing a name are compared: merged when they share a co-author or institution (or are in `MANUAL_SAME_PAIRS`), kept apart when full first names differ or countries differ with nothing shared, and left separate when unclear.

Writes `authorships_raw.csv` (stage 2), `dblp_author_matches.csv`, `gephi/duplicate_author_decisions.csv`, `authorships.csv`, `gephi/coauthorship_network.gexf`, `gephi/coauthorship_nodes.csv`, `gephi/coauthorship_edges.csv`, and `network_missing_papers.csv`.

In [ ]:
import itertools, unicodedata, networkx as nx
m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
A = pd.read_csv(os.path.join(REV, 'authorships_stage1.csv'))
def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]+', ' ', s)).strip()
def key_full(n):
    t = fold(str(n).translate(DASHES)).split()   # Unicode hyphens (e.g. "Luna\u2010Reyes") count as hyphens
    return f'{t[-1]}_{t[0][:1]}' if len(t) >= 2 else ''
# Stricter name matching (used by Step 11). A name matches an OpenAlex author only when the first initials agree,
# full first names (3+ letters) do not conflict, and the full surname (hyphenated parts joined, e.g. "Gil-Garcia" ->
# "gilgarcia") appears at the end of BOTH names. So "Jaime Garcia" no longer matches "J. Ramon Gil-Garcia",
# while "J. R. Gil-Garcia", "David Valle Cruz" or "M. Meyerhoff Nielsen" still find their OpenAlex record.
DASHES = dict.fromkeys(map(ord, '‐‑‒–—−'), '-')
def nm_tokens(n):
    s = unicodedata.normalize('NFKD', str(n).translate(DASHES)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z\- ]+', ' ', s).split()
def nm_parts(n):
    t = nm_tokens(n)
    if len(t) < 2: return None
    flat = [x for w in t for x in w.split('-') if x]
    sur = re.sub('-', '', t[-1])
    tails = {''.join(flat[i:]) for i in range(1, len(flat))}
    return flat[0][:1], flat[0], sur, tails
def names_match(a, b):
    pa, pb = nm_parts(a), nm_parts(b)
    if not pa or not pb or pa[0] != pb[0]: return False
    fa, fb = pa[1], pb[1]
    if len(fa) > 2 and len(fb) > 2 and not (fa.startswith(fb) or fb.startswith(fa)): return False
    lc = max(pa[2], pb[2], key=len)
    return lc in pa[3] and lc in pb[3]

idx = {}
for r in A[~A.author_key.str.startswith('name:')].drop_duplicates('author_key').itertuples():
    idx.setdefault(key_full(r.author_name), set()).add(r.author_key)
oa_names = A[~A.author_key.str.startswith('name:')].groupby('author_key').author_name.apply(set).to_dict()
def oa_hits(n):
    return [o for o in idx.get(key_full(n), set()) if any(names_match(n, x) for x in oa_names.get(o, ()))]
def resolve(n):
    k = key_full(n); c = oa_hits(n)
    return (c[0], 'name_matched_to_openalex') if len(c) == 1 else (f'name:{k}', 'name_only')

miss = m[~m.row_id.isin(A.row_id)]
D['ft'] = D.title.map(fold)
new, log = [], []
for p in miss.itertuples():
    pt = fold(re.sub(r'\s+\d{1,2}(st|nd|rd|th) Annual International Conference.*$', '', str(p.title)))
    cand = D[(D.year - p.year).abs() <= 1]
    best, bs = None, 0
    for c in cand.itertuples():
        s = fuzz.token_sort_ratio(pt, c.ft)
        if s < 90 and min(len(pt), len(c.ft)) >= 25:
            first = fold(str(p.authors).split(',')[0]).split()
            if fuzz.partial_ratio(pt, c.ft) >= 95 and first and first[-1] in fold(' '.join(c.authors)).split():
                s = fuzz.partial_ratio(pt, c.ft)
        if s > bs: best, bs = c, s
    ok = best is not None and bs >= 90
    if ok:
        for k, n in enumerate(best.authors, 1):
            key, how = resolve(n)
            new.append(dict(row_id=p.row_id, order=k, author_key=key, author_name=n, institution=None, country=None,
                            author_source=f'dblp ({how})', year=p.year, topic_name_k10=p.topic_name_k10))
    log.append(dict(row_id=p.row_id, year=p.year, dgrl_title=p.title, dblp_title=best.title if ok else '',
                    score=bs, dblp_key=best.key if ok else '', dblp_link=best.ee if ok else ''))
L = pd.DataFrame(log); L.to_csv(os.path.join(REV, 'dblp_author_matches.csv'), index=False)
A = pd.concat([A, pd.DataFrame(new)], ignore_index=True)

# Web lookups give author lists only for papers that OpenAlex, Crossref, DGRL and DBLP could not list
# (their lists can be partial, so every fuller source wins; Step 12b attaches their locations to the other lists by name)
wl = pd.read_csv(AFF_LOG)
wl = wl[~wl.row_id.isin(ORIG_LOG_IDS) & ~wl.row_id.isin(A.row_id)].drop_duplicates(['row_id', 'author_order'])
yt = m.set_index('row_id')[['year', 'topic_name_k10']]
web_rows = []
for a in wl.sort_values(['row_id', 'author_order']).itertuples():
    key, how = resolve(a.author)
    web_rows.append(dict(row_id=a.row_id, order=a.author_order, author_key=key, author_name=a.author, institution=a.institution,
                         country=a.country, author_source=f'web_lookup_log ({how})', year=yt.year.get(a.row_id), topic_name_k10=yt.topic_name_k10.get(a.row_id)))
A = pd.concat([A, pd.DataFrame(web_rows)], ignore_index=True)
print(f"Papers added from the web lookups (no other author list): {wl.row_id.nunique()}")

# Authors printed on the paper but missing from the OpenAlex/DBLP list (found while checking co-authors by hand).
# Added only when no author with the same surname and first initial is already listed for that paper.
ADD_MISSING_AUTHORS = True
co_path = os.path.join(IN, 'coauthor_affiliation_lookups_2026-11-20.csv')
if ADD_MISSING_AUTHORS and os.path.exists(co_path):
    co = pd.read_csv(co_path, keep_default_na=False)
    co = co[co.row_id.isin(A.row_id) & co.source_type.str.contains('title page|Contributors panel', case=False)]
    have_k = A.groupby('row_id').author_name.apply(lambda s: {key_full(n) for n in s})
    extra = []
    for a in co.itertuples():
        if key_full(a.author) in have_k.get(a.row_id, set()): continue
        key, how = resolve(a.author)
        extra.append(dict(row_id=a.row_id, order=a.author_order, author_key=key, author_name=a.author, institution=a.institution,
                          country=a.country, author_source=f'paper_author_list ({how})', year=yt.year.get(a.row_id), topic_name_k10=yt.topic_name_k10.get(a.row_id)))
    A = pd.concat([A, pd.DataFrame(extra)], ignore_index=True)
    print(f"Authors added from the papers' own author lists: {len(extra)} on {len({e['row_id'] for e in extra})} papers")

# Name-only authors (from DGRL citations or DBLP) are linked to an OpenAlex author when exactly one matches by the rule
# above AND there is supporting evidence: a shared institution, a shared co-author, or a hyphenated surname.
inst_of = A.dropna(subset=['institution']).groupby('author_key').institution.apply(lambda s: {fold(x) for x in s}).to_dict()
co_of = {}
for rid, g in A.groupby('row_id'):
    ks = set(g.author_key)
    for k in ks: co_of.setdefault(k, set()).update(ks - {k})
relinked = []
for k, g in A[A.author_key.str.startswith('name:')].groupby('author_key'):
    n = g.author_name.iloc[0]; hits = oa_hits(n)
    if len(hits) != 1: continue
    o = hits[0]
    ev = (inst_of.get(k, set()) & inst_of.get(o, set())) or ((co_of.get(k, set()) & co_of.get(o, set())) - {k, o}) or ('-' in nm_tokens(n)[-1])
    if ev:
        A.loc[A.author_key == k, 'author_source'] += ' + relinked to openalex'
        A.loc[A.author_key == k, 'author_key'] = o
        relinked.append(dict(name_key=k, name=n, openalex_id=o, openalex_name=sorted(oa_names[o])[0], papers=g.row_id.nunique()))
pd.DataFrame(relinked).to_csv(os.path.join(REV, 'name_only_relinked.csv'), index=False)
print(f"Name-only authors linked to their OpenAlex record: {len(relinked)} (see name_only_relinked.csv)")
A.to_csv(os.path.join(REV, 'authorships_raw.csv'), index=False)
print(f"\nMissing papers matched in DBLP: {(L.dblp_key != '').sum()} of {len(miss)}")

oa = A[~A.author_key.str.startswith('name:')]
nodes0 = oa.groupby('author_key').author_name.agg(lambda s: s.mode().iloc[0]).reset_index()
nodes0['nk'] = nodes0.author_name.map(key_full)
cands = nodes0[nodes0.duplicated('nk', keep=False) & (nodes0.nk != '')]
coauth = {}
for rid, g in A.groupby('row_id'):
    ks = set(g.author_key)
    for k in ks: coauth.setdefault(k, set()).update(ks - {k})
insts = A.dropna(subset=['institution']).groupby('author_key').institution.apply(lambda s: {fold(x) for x in s})
ctry = A.dropna(subset=['country']).groupby('author_key').country.apply(set)
names = A.groupby('author_key').author_name.apply(set)
firsts = lambda k: {fold(n).split()[0] for n in names.get(k, []) if fold(n)}
def conflict(a, b):
    fa, fb = {f for f in firsts(a) if len(f) > 2}, {f for f in firsts(b) if len(f) > 2}
    return bool(fa and fb) and not any(x == y or x.startswith(y) or y.startswith(x) for x in fa for y in fb)
MANUAL_SAME = {frozenset(p) for p in MANUAL_SAME_PAIRS}
dec = []
for nk, g in cands.groupby('nk'):
    for a, b in itertools.combinations(g.author_key, 2):
        na, nb = ' / '.join(sorted(names[a])), ' / '.join(sorted(names[b]))
        sc = (coauth.get(a, set()) & coauth.get(b, set())) - {a, b}
        si = insts.get(a, set()) & insts.get(b, set())
        dc = bool(ctry.get(a)) and bool(ctry.get(b)) and not (ctry.get(a) & ctry.get(b))
        if frozenset((na, nb)) in MANUAL_SAME: v, why = 'likely_same', 'manual review: name variant'
        elif conflict(a, b): v, why = 'likely_different', 'different full first names'
        elif sc or si: v, why = 'likely_same', f'shared co-authors: {len(sc)}; shared institution: {bool(si)}'
        elif dc: v, why = 'likely_different', 'different countries, nothing shared'
        else: v, why = 'unclear', 'no evidence either way'
        dec.append(dict(name_key=nk, id_a=a, name_a=na, id_b=b, name_b=nb, verdict=v, evidence=why))
Dd = pd.DataFrame(dec)
papers = A.groupby('author_key').row_id.nunique()
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for r in Dd[Dd.verdict == 'likely_same'].itertuples():
    ra, rb = find(r.id_a), find(r.id_b)
    if ra != rb:
        keep, drop = (ra, rb) if papers.get(ra, 0) >= papers.get(rb, 0) else (rb, ra)
        parent[drop] = keep
remap = {k: find(k) for k in list(parent) if find(k) != k}
Dd.to_csv(os.path.join(OUT, 'duplicate_author_decisions.csv'), index=False)
A['author_key_original'] = A.author_key
A['author_key'] = A.author_key.map(lambda k: remap.get(k, k))
A = A.drop_duplicates(['row_id', 'author_key'])
A.to_csv(os.path.join(REV, 'authorships.csv'), index=False)

PH = [(a, b) for a, b, _ in PHASES]
phase = lambda y: next(f'{a}-{b}' for a, b in PH if a <= y <= b)
mode = lambda s: s.dropna().mode().iloc[0] if s.notna().any() else ''
nodes = A.groupby('author_key').agg(Label=('author_name', mode), papers=('row_id', 'nunique'), first_year=('year', 'min'),
          last_year=('year', 'max'), country=('country', mode), institution=('institution', mode), main_topic=('topic_name_k10', mode),
          id_type=('author_source', lambda s: 'name_only' if s.str.contains('name_only').all() else 'openalex')).reset_index().rename(columns={'author_key': 'Id'})
nodes['first_phase'] = nodes.first_year.map(phase)
pairs = [(a, b, rid, int(g.year.iloc[0])) for rid, g in A.groupby('row_id') for a, b in itertools.combinations(sorted(set(g.author_key)), 2)]
edges = pd.DataFrame(pairs, columns=['Source', 'Target', 'row_id', 'year']).groupby(['Source', 'Target']).agg(
          Weight=('row_id', 'nunique'), first_year=('year', 'min'), last_year=('year', 'max')).reset_index()
edges['Type'] = 'Undirected'
nodes.to_csv(os.path.join(OUT, 'coauthorship_nodes.csv'), index=False)
edges.to_csv(os.path.join(OUT, 'coauthorship_edges.csv'), index=False)
G = nx.Graph()
for r in nodes.itertuples():
    G.add_node(r.Id, label=r.Label, papers=int(r.papers), first_year=int(r.first_year), last_year=int(r.last_year), country=str(r.country),
               institution=str(r.institution), main_topic=str(r.main_topic), first_phase=r.first_phase, id_type=r.id_type)
for r in edges.itertuples():
    G.add_edge(r.Source, r.Target, weight=int(r.Weight), first_year=int(r.first_year), last_year=int(r.last_year))
nx.write_gexf(G, os.path.join(OUT, 'coauthorship_network.gexf'))

comps = sorted(nx.connected_components(G), key=len, reverse=True)
left = m[~m.row_id.isin(A.row_id)]
print(f"\n===== FINAL =====")
print(f"Papers in network: {A.row_id.nunique()} of {len(m)} ({A.row_id.nunique()/len(m):.1%}) | still missing: {len(left)}")
print(f"Authors: {G.number_of_nodes()} | Links: {G.number_of_edges()} | IDs merged: {len(remap)}")
print(f"Largest connected group: {len(comps[0])} ({len(comps[0])/G.number_of_nodes():.0%})")
print("\nTop 10 most collaborative:")
for n, d in sorted(G.degree(weight='weight'), key=lambda x: -x[1])[:10]:
    print(f"  {G.nodes[n]['label']}: {d} ({G.nodes[n]['institution']})")
print("\nStill missing by year:", left.groupby('year').size().to_dict())
left[['row_id', 'year', 'title', 'authors', 'doi']].to_csv(os.path.join(REV, 'network_missing_papers.csv'), index=False)
print("Saved: gephi/coauthorship_network.gexf (+ nodes, edges, duplicate log), dblp_author_matches.csv")

## Step 12. Author-level locations from Crossref
About 110 papers had only paper-level countries. Their Crossref author lists are re-read, each affiliation matched through ROR, and attached to the matching author (by surname and first initial, or by author order when the lists line up exactly).

Updates `authorships_raw.csv` and `authorships.csv` (adds `+ crossref/ror location` to `author_source`) and writes `author_location_recovery_log.csv`.

In [ ]:
import os, re, json, time, shutil, unicodedata, urllib.parse, requests, pandas as pd

m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
files = {k: os.path.join(REV, f) for k, f in [('raw', 'authorships_raw.csv'), ('dedup', 'authorships.csv')]}
A = pd.read_csv(files['dedup'])

has_author_loc = set(A.dropna(subset=['country']).row_id)
targets = m[(m.n_countries.fillna(0) > 0) & ~m.row_id.isin(has_author_loc)]
print(f"Papers with paper-level countries but no author-level locations: {len(targets)}")

def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z ]+', ' ', s)).strip()
def nkey(first, last):
    f, l = fold(first), fold(last).replace(' ', '')
    return f'{l}_{f[:1]}' if l else ''
def nkey_full(n):
    t = fold(n).split()
    return nkey(t[0], t[-1]) if len(t) >= 2 else ''

updates, log = [], []
for p in targets.itertuples():
    doi = p.doi if isinstance(p.doi, str) and p.doi else None
    if not doi:
        log.append(dict(row_id=p.row_id, status='no DOI')); continue
    js = get_json(f'https://api.crossref.org/works/{urllib.parse.quote(doi)}', {'mailto': EMAIL}); pause(0.2)
    authors = ((js or {}).get('message') or {}).get('author', []) or []
    cr = []
    for k, a in enumerate(authors, 1):
        ms = [x for x in (ror_match(af.get('name', '')) for af in a.get('affiliation', [])) if x and x.get('country')]
        if ms: cr.append(dict(order=k, key=nkey(a.get('given', ''), a.get('family', '')), inst=ms[0]['name'], cc=ms[0]['country']))
    if not cr:
        log.append(dict(row_id=p.row_id, status='Crossref has no per-author affiliations')); continue
    rows = A[A.row_id == p.row_id]
    matched = 0
    for c in cr:
        hit = rows[rows.author_name.map(nkey_full) == c['key']]
        if len(hit) != 1 and len(rows) == len(authors):            # fall back to author order when lists line up
            hit = rows[rows['order'] == c['order']]
        if len(hit) == 1:
            updates.append((p.row_id, hit.iloc[0].author_key, c['inst'], c['cc'])); matched += 1
    log.append(dict(row_id=p.row_id, status=f'located {matched} of {len(rows)} authors'))
json.dump(ror_cache, open(ROR_CACHE_PATH, 'w'))

U = pd.DataFrame(updates, columns=['row_id', 'author_key', 'inst', 'cc'])
for name, path in files.items():
    X = pd.read_csv(path)
    keycol = 'author_key'
    X = X.merge(U, on=['row_id', keycol], how='left')
    fill = X.country.isna() & X.cc.notna()
    X.loc[fill, 'country'] = X.loc[fill, 'cc']; X.loc[fill, 'institution'] = X.loc[fill, 'inst']
    X.loc[fill, 'author_source'] = X.loc[fill, 'author_source'].astype(str) + ' + crossref/ror location'
    X.drop(columns=['inst', 'cc']).to_csv(path, index=False)
    print(f"{os.path.basename(path)}: filled {fill.sum()} author locations")

L = pd.DataFrame(log); L.to_csv(os.path.join(REV, 'author_location_recovery_log.csv'), index=False)
A2 = pd.read_csv(files['dedup'])
print(f"\nPapers with author-level locations: {A2.dropna(subset=['country']).row_id.nunique()} (was {len(has_author_loc)})")
print(L.status.str.replace(r'\d+ of \d+', 'N', regex=True).value_counts().to_string())

save_api_cache()

## Step 12b. Author-level locations from the affiliation logs
Papers that are in the network but whose authors still have no location get them from the combined affiliation log, matched by surname and first initial, or by author order when the lists line up exactly. Authors with two affiliations keep the first one listed (the network uses one place per author per paper).

Updates `authorships_raw.csv` and `authorships.csv` (adds `+ manual_log location` to `author_source`).


In [ ]:
import os, re, unicodedata, pandas as pd

m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
mlog = pd.read_csv(AFF_LOG)
mlog = mlog.sort_values(['row_id', 'author_order'], kind='stable').drop_duplicates(['row_id', 'author_order'])   # first listed affiliation per author

def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z ]+', ' ', s)).strip()
def nkey(n):
    t = fold(n).split()
    return f"{t[-1]}_{t[0][:1]}" if len(t) >= 2 else ''

files = {k: os.path.join(REV, f) for k, f in [('raw', 'authorships_raw.csv'), ('dedup', 'authorships.csv')]}
before = pd.read_csv(files['dedup']).dropna(subset=['country']).row_id.nunique()
for name, path in files.items():
    X = pd.read_csv(path)
    filled = 0
    for rid, g in mlog.groupby('row_id'):
        rows = X[(X.row_id == rid) & X.country.isna()]
        if rows.empty: continue
        n_all = (X.row_id == rid).sum()
        for a in g.itertuples():
            hit = rows[rows.author_name.map(nkey) == nkey(a.author)]
            if len(hit) != 1 and n_all == len(g):                     # fall back to author order when the lists line up
                hit = rows[rows['order'] == a.author_order]
            if len(hit) == 1:
                i = hit.index[0]
                X.loc[i, ['country', 'institution']] = [a.country, a.institution]
                X.loc[i, 'author_source'] = str(X.loc[i, 'author_source']) + ' + manual_log location'
                rows = rows.drop(i); filled += 1
    X.to_csv(path, index=False)
    print(f"{os.path.basename(path)}: filled {filled} author locations from the affiliation log")
after = pd.read_csv(files['dedup']).dropna(subset=['country']).row_id.nunique()
print(f"Papers with author-level locations: {after} (was {before})")


## Step 13. Country and institution networks
Each author pair on a paper becomes an undirected link between their countries (or institutions). Edge `Weight` is the number of papers linking two places; `author_pairs` counts author pairs; per-phase paper counts are included. Same-place links (e.g. US-US) are included everywhere: as rows with `same_place = True` in the CSVs and as self-loops in the Gephi files. Each node also has `domestic_papers`, the number of papers with two or more authors from that place.

Writes `gephi/country_network.gexf`, `gephi/country_nodes.csv`, `gephi/country_edges.csv`, and the same three for institutions.

In [ ]:
import os, itertools, pandas as pd, networkx as nx, pycountry_convert as pc
A = pd.read_csv(os.path.join(REV, 'authorships.csv'))

PH = [(a, b) for a, b, _ in PHASES]
phase = lambda y: next(f'{a}-{b}' for a, b in PH if a <= y <= b)
def region(cc):
    if cc in ('US', 'CA'): return 'North America'
    try: return {'NA': 'Latin America', 'SA': 'Latin America', 'EU': 'Europe', 'AS': 'Asia',
                 'AF': 'Africa', 'OC': 'Oceania'}[pc.country_alpha2_to_continent_code(cc)]
    except Exception: return ''

def build(level):
    col = 'country' if level == 'country' else 'institution'
    X = A.dropna(subset=[col]).drop_duplicates(['row_id', 'author_key'])   # one place per author per paper
    X = X[X[col].astype(str).str.strip() != '']
    rows = []
    for rid, g in X.groupby('row_id'):
        yr = int(g.year.iloc[0]); places = g[col].tolist()
        for a, b in itertools.combinations(places, 2):        # every author pair on the paper
            s, t = sorted([a, b])
            rows.append((s, t, rid, yr))
    E = pd.DataFrame(rows, columns=['Source', 'Target', 'row_id', 'year'])
    E['phase'] = E.year.map(phase)
    edges = E.groupby(['Source', 'Target']).agg(
        author_pairs=('row_id', 'size'), Weight=('row_id', 'nunique'),
        first_year=('year', 'min'), last_year=('year', 'max')).reset_index()
    for p in [f'{a}-{b}' for a, b in PH]:
        cnt = E[E.phase == p].groupby(['Source', 'Target']).row_id.nunique().rename(f'papers_{p}')
        edges = edges.merge(cnt, on=['Source', 'Target'], how='left')
    edges = edges.fillna(0)
    edges['same_place'] = edges.Source == edges.Target
    edges['Type'] = 'Undirected'

    nodes = X.groupby(col).agg(papers=('row_id', 'nunique'), authors=('author_key', 'nunique'),
                               first_year=('year', 'min')).reset_index().rename(columns={col: 'Id'})
    nodes['Label'] = nodes.Id
    if level == 'country':
        nodes['region'] = nodes.Id.map(region)
    else:
        cmap = A.dropna(subset=['institution', 'country']).groupby('institution').country.agg(lambda s: s.mode().iloc[0])
        nodes['country'] = nodes.Id.map(cmap).fillna('')
        nodes['region'] = nodes.country.map(region)
    nodes.to_csv(os.path.join(OUT, f'{level}_nodes.csv'), index=False)
    edges.to_csv(os.path.join(OUT, f'{level}_edges.csv'), index=False)

    G = nx.Graph()
    for r in nodes.itertuples():
        G.add_node(r.Id, label=str(r.Label), papers=int(r.papers), authors=int(r.authors), first_year=int(r.first_year),
                   region=str(r.region), **({'country': str(r.country)} if level == 'institution' else {}))
    dom = edges[edges.same_place].set_index('Source').Weight
    for n in G.nodes:
        G.nodes[n]['domestic_papers'] = int(dom.get(n, 0))   # papers with 2+ authors from the same place
    for r in edges.itertuples():                          # all links, including same-place links (e.g. US-US) as self-loops
        G.add_edge(r.Source, r.Target, weight=int(r.Weight), author_pairs=int(r.author_pairs), first_year=int(r.first_year))
    nx.write_gexf(G, os.path.join(OUT, f'{level}_network.gexf'))
    return X, edges, G

for level in ['country', 'institution']:
    plural = 'countries' if level == 'country' else 'institutions'
    X, edges, G = build(level)
    cross = edges[~edges.same_place]
    print(f"\n===== {level.upper()} NETWORK =====")
    print(f"Papers with {level} data: {X.row_id.nunique()} | {level.title()} nodes: {G.number_of_nodes()} | links between different {plural}: {G.number_of_edges()}")
    print(f"Within-{level} collaborations (e.g., US-US): {edges[edges.same_place].Weight.sum():.0f} paper-links | across {plural}: {cross.Weight.sum():.0f}")
    print(f"\nTop 10 cross-{level} links (number of papers):")
    print(cross.sort_values('Weight', ascending=False).head(10)[['Source', 'Target', 'Weight']].to_string(index=False))
    if level == 'country':
        print("\nShare of country links that are international, by phase:")
        for p in [f'{a}-{b}' for a, b in PH]:
            tot = edges[f'papers_{p}'].sum(); intl = cross[f'papers_{p}'].sum()
            print(f"  {p}: {intl:.0f} of {tot:.0f} ({intl/tot:.0%})" if tot else f"  {p}: no data")
print("\nSaved in gephi/: country_network.gexf, country_nodes.csv, country_edges.csv, institution_network.gexf, institution_nodes.csv, institution_edges.csv")

## Step 14. World maps and collaboration rankings
Draws the country network on an Equal Earth world map for three cumulative ranges (2000 to 2005, 2000 to 2012, 2000 to 2024), all on the same circle and line scales so they can be compared. Every country is labelled with its full name: Europe, too crowded at world scale, gets a zoomed inset, and overlapping labels are moved apart with adjustText. Each range is rebuilt from `authorships.csv` exactly as in Step 13; the full range is checked against `country_network.gexf`. Country outlines are Natural Earth 1:110m (public domain), pinned to release v5.1.2 and saved to `geo/`.

Also ranks countries by partner countries (how widely they collaborate) and by internationally co-authored papers, and ranks country pairs by joint papers and number of topics, for each phase and the cumulative ranges.

Writes `gephi/country_collaboration_map_2000_2005.png`, `gephi/country_collaboration_map_2000_2012.png`, `gephi/country_collaboration_map.png`, `gephi/country_collaboration_rankings.csv` and `gephi/country_pair_rankings.csv`.

In [ ]:
import os, json, math, hashlib, itertools, requests, subprocess, sys
try:
    from adjustText import adjust_text        # nudges labels apart so every country can be labelled
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'adjustText==1.4.0'], check=True)
    from adjustText import adjust_text
import pandas as pd, networkx as nx
import matplotlib, matplotlib.pyplot as plt, matplotlib.patheffects
from matplotlib.patches import Polygon, PathPatch
from matplotlib.path import Path
from matplotlib.lines import Line2D

# Natural Earth 1:110m country outlines (public domain), pinned to release v5.1.2 and kept in the project folder
GEO_URL = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/v5.1.2/geojson/ne_110m_admin_0_countries.geojson'
GEO = os.path.join(REV, 'geo', 'ne_110m_admin_0_countries.geojson')
os.makedirs(os.path.dirname(GEO), exist_ok=True)
if not os.path.exists(GEO):
    r = requests.get(GEO_URL, timeout=60); r.raise_for_status(); open(GEO, 'wb').write(r.content)
print('Natural Earth md5:', hashlib.md5(open(GEO, 'rb').read()).hexdigest())   # 21abb20e7c74fc0c66b8dd955f8f7f73

W = json.load(open(GEO))
G_all = nx.read_gexf(os.path.join(OUT, 'country_network.gexf'))
A = pd.read_csv(os.path.join(REV, 'authorships.csv'), keep_default_na=False)
m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'), keep_default_na=False)
B = A[A.country != ''].copy()
B['topic'] = B.row_id.map(dict(zip(m.row_id, m.topic_k10.astype(int))))
REGION_OF = {n: a['region'] for n, a in G_all.nodes(data=True)}

def network(y0, y1):
    """Country network for papers published y0..y1, built exactly like Step 13 (author-level locations)."""
    S = B[(B.year >= y0) & (B.year <= y1)]
    papers = S.groupby('country').row_id.nunique()
    W_, topics = {}, {}
    for rid, g in S.groupby('row_id'):
        seen = set()
        for (k1, c1), (k2, c2) in itertools.combinations(zip(g.author_key, g.country), 2):
            if k1 != k2: seen.add(tuple(sorted((c1, c2))))
        for e in seen:
            W_[e] = W_.get(e, 0) + 1; topics.setdefault(e, set()).add(int(g.topic.iloc[0]))
    return papers, W_, topics

# check: the full range reproduces country_network.gexf exactly
p_all, w_all, _ = network(2000, 2024)
assert {tuple(sorted((u, v))): int(d['weight']) for u, v, d in G_all.edges(data=True)} == w_all
assert all(int(G_all.nodes[n]['papers']) == p_all[n] for n in G_all.nodes)

# ---------- maps: one per cumulative range, all on the same scales so they can be compared ----------
REGION_COLOR = {'North America': '#2a78d6', 'Europe': '#eb6834', 'Latin America': '#1baf7a',
                'Asia': '#eda100', 'Oceania': '#e87ba4', 'Africa': '#008300'}
INK, MUTED, LAND, EDGE_C, BG = '#1f1f1e', '#6b6a64', '#e6e5e0', '#4a4a47', '#fcfcfb'
A1, A2, A3, A4 = 1.340264, -0.081106, 0.000893, 0.003796
M = math.sqrt(3) / 2
def proj(lon, lat):   # Equal Earth projection (keeps areas honest)
    l, p = math.radians(lon), math.radians(lat)
    t = math.asin(M * math.sin(p)); t2 = t * t; t6 = t2 ** 3
    return l * math.cos(t) / (M * (A1 + 3 * A2 * t2 + t6 * (7 * A3 + 9 * A4 * t2))), t * (A1 + A2 * t2 + t6 * (A3 + A4 * t2))
pt, NAME, LL = {}, {}, {}
for f in W['features']:
    pr = f['properties']; iso = pr.get('ISO_A2_EH') or pr.get('ISO_A2')
    if pr.get('NAME') != 'Antarctica' and iso and iso != '-99' and pr.get('LABEL_X') is not None:
        pt[iso] = proj(pr['LABEL_X'], pr['LABEL_Y']); NAME[iso] = pr.get('NAME_EN') or pr['NAME']; LL[iso] = (pr['LABEL_X'], pr['LABEL_Y'])
for iso, (lon, lat) in {'SG': (103.8, 1.35), 'HK': (114.17, 22.32), 'BN': (114.7, 4.5), 'LU': (6.13, 49.6),
                        'MO': (113.55, 22.2), 'SI': (14.8, 46.1), 'MT': (14.4, 35.9)}.items():
    pt.setdefault(iso, proj(lon, lat)); LL.setdefault(iso, (lon, lat))
SHORT_NAME = {'US': 'United States', 'CN': 'China', 'KR': 'South Korea', 'GB': 'United Kingdom', 'RU': 'Russia', 'CZ': 'Czechia',
              'AE': 'UAE', 'HK': 'Hong Kong', 'SG': 'Singapore', 'LU': 'Luxembourg', 'SI': 'Slovenia', 'MT': 'Malta', 'TW': 'Taiwan', 'BN': 'Brunei', 'MO': 'Macao', 'NL': 'Netherlands'}
EU_BOX = (-11, 33, 35, 61)                                       # lon/lat box shown in the Europe inset
PMAX = int(p_all.max())                                          # fixed scales from the full 2000 to 2024 network
WMAX = max(w for (u, v), w in w_all.items() if u != v)           # strongest cross-country link 
size = lambda p: 18 + 1400 * (p / PMAX)

def draw(y0, y1, out):
    papers, wts, _ = network(y0, y1)
    cross = {e: w for e, w in wts.items() if e[0] != e[1]}
    dom_us = wts.get(('US', 'US'), 0)
    fig, ax = plt.subplots(figsize=(16, 8.6), dpi=150)
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for f in W['features']:
        if f['properties'].get('NAME') == 'Antarctica': continue
        g = f['geometry']
        for poly in (g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]):
            ax.add_patch(Polygon([proj(x, y) for x, y in poly[0]], closed=True, fc=LAND, ec=BG, lw=0.6, zorder=1))
    for (u, v), w in sorted(cross.items(), key=lambda kv: kv[1]):
        (x1, y1_), (x2, y2) = pt[u], pt[v]; dx, dy = x2 - x1, y2 - y1_
        path = Path([(x1, y1_), ((x1 + x2) / 2 - dy * 0.18, (y1_ + y2) / 2 + dx * 0.18), (x2, y2)], [Path.MOVETO, Path.CURVE3, Path.CURVE3])
        ax.add_patch(PathPatch(path, fc='none', ec=EDGE_C, lw=0.6 + 3.4 * (w / WMAX), alpha=0.25 + 0.55 * (w / WMAX), capstyle='round', zorder=2))
    nodes = sorted(papers.index, key=lambda n: -papers[n])
    for n in nodes:
        ax.scatter(*pt[n], s=size(papers[n]), c=REGION_COLOR.get(REGION_OF.get(n), MUTED), edgecolors=BG, linewidths=1.6, zorder=3)
    ax.set_xlim(proj(-180, 0)[0], proj(180, 0)[0]); ax.set_ylim(proj(0, -58)[1], proj(0, 84)[1]); ax.set_aspect('equal'); ax.axis('off')
    # every country is labelled. Europe is too crowded at world scale, so it gets a zoomed inset (in the empty
    # Indian Ocean) where its countries are labelled; elsewhere labels sit beside the dots. adjustText nudges
    # overlapping labels apart and draws a thin leader line when a label had to move.
    use_names = True                                   # full country names on every map
    fs = 9 if len(nodes) <= 25 else 7.6                 # a little smaller on the crowded full map
    label = lambda n: SHORT_NAME.get(n, NAME.get(n, n)) if use_names else n
    halo = [matplotlib.patheffects.withStroke(linewidth=2.5, foreground=BG)]
    in_eu = lambda n: EU_BOX[0] <= LL[n][0] <= EU_BOX[1] and EU_BOX[2] <= LL[n][1] <= EU_BOX[3]
    outside = [n for n in nodes if not in_eu(n)]
    t_out = [ax.text(pt[n][0], pt[n][1], label(n), fontsize=fs, color=INK, fontweight='bold', zorder=5, va='center', path_effects=halo) for n in outside]
    if t_out:
        adjust_text(t_out, x=[pt[n][0] for n in outside], y=[pt[n][1] for n in outside], ax=ax, expand=(1.25, 1.5),
                    force_text=(0.2, 0.4), force_static=(0.3, 0.5), ensure_inside_axes=True,
                    arrowprops=dict(arrowstyle='-', color=MUTED, lw=0.6, shrinkA=2, shrinkB=3))
    box = [proj(lon, EU_BOX[2]) for lon in range(EU_BOX[0], EU_BOX[1] + 1)] + [proj(lon, EU_BOX[3]) for lon in range(EU_BOX[1], EU_BOX[0] - 1, -1)]
    ax.add_patch(Polygon(box, closed=True, fc='none', ec=MUTED, lw=0.8, ls=(0, (3, 2)), zorder=4))
    fig.subplots_adjust(left=0.01, right=0.79, top=0.91, bottom=0.04)   # world map on the left, Europe inset in the right margin
    ins = fig.add_axes([0.795, 0.08, 0.20, 0.62])
    ins.set_facecolor(BG)
    for f in W['features']:
        if f['properties'].get('NAME') == 'Antarctica': continue
        g = f['geometry']
        for poly in (g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]):
            ins.add_patch(Polygon([proj(x, y) for x, y in poly[0]], closed=True, fc=LAND, ec=BG, lw=0.6, zorder=1))
    eu = [n for n in nodes if in_eu(n)]
    for (u, v), w in sorted(cross.items(), key=lambda kv: kv[1]):
        if u in eu and v in eu:
            (x1, y1_), (x2, y2) = pt[u], pt[v]; dx, dy = x2 - x1, y2 - y1_
            path = Path([(x1, y1_), ((x1 + x2) / 2 - dy * 0.18, (y1_ + y2) / 2 + dx * 0.18), (x2, y2)], [Path.MOVETO, Path.CURVE3, Path.CURVE3])
            ins.add_patch(PathPatch(path, fc='none', ec=EDGE_C, lw=0.6 + 3.4 * (w / WMAX), alpha=0.25 + 0.55 * (w / WMAX), capstyle='round', zorder=2))
    for n in eu:
        ins.scatter(*pt[n], s=size(papers[n]), c=REGION_COLOR['Europe'], edgecolors=BG, linewidths=1.4, zorder=3)
    xs = [x for x, _ in box]; ys = [y for _, y in box]
    ins.set_xlim(min(xs), max(xs)); ins.set_ylim(min(ys), max(ys)); ins.set_aspect('equal', adjustable='box'); ins.set_anchor('C')   # panel shrinks to fit the Europe box exactly
    ins.set_xticks([]); ins.set_yticks([])
    for s in ins.spines.values(): s.set_edgecolor(MUTED); s.set_linewidth(0.8)
    ins.set_title('Europe (links within Europe)', fontsize=9, color=MUTED, loc='left', pad=3)
    t_eu = [ins.text(pt[n][0], pt[n][1], label(n), fontsize=fs - 0.4, color=INK, fontweight='bold', zorder=5, va='center', path_effects=halo) for n in eu]
    if t_eu:
        adjust_text(t_eu, x=[pt[n][0] for n in eu], y=[pt[n][1] for n in eu], ax=ins, expand=(1.3, 1.6),
                    force_text=(0.3, 0.5), force_static=(0.4, 0.6), ensure_inside_axes=True,
                    arrowprops=dict(arrowstyle='-', color=MUTED, lw=0.5, shrinkA=2, shrinkB=3))
    leg1 = ax.legend(handles=[Line2D([0], [0], marker='o', ls='', ms=9, mfc=c, mec=BG, mew=1.2, label=r) for r, c in REGION_COLOR.items()],
                     title='Region', loc='lower left', frameon=False, fontsize=10, title_fontsize=10.5, labelcolor=INK, bbox_to_anchor=(0.0, 0.02))
    ax.add_artist(leg1)
    leg2 = ax.legend(handles=[Line2D([0], [0], marker='o', ls='', ms=math.sqrt(size(p)), mfc='none', mec=MUTED, label=f'{p} papers') for p in (10, 50, 265)],
                     title='Country size', loc='lower left', frameon=False, fontsize=10, title_fontsize=10.5, labelcolor=INK,
                     bbox_to_anchor=(0.13, 0.02), labelspacing=1.6, borderpad=1, handletextpad=2.2)
    ax.add_artist(leg2)
    ax.legend(handles=[Line2D([0], [0], color=EDGE_C, lw=0.6 + 3.4 * (w / WMAX), alpha=0.25 + 0.55 * (w / WMAX), label=f'{w} paper{"s" if w > 1 else ""}') for w in sorted({1, max(2, round(WMAX / 2)), WMAX})],
              title='Link between countries', loc='lower left', frameon=False, fontsize=10, title_fontsize=10.5, labelcolor=INK, bbox_to_anchor=(0.0, 0.34))   # above the region key, clear of South America
    ax.set_title(f'International collaboration among dg.o authors, {y0} to {y1}', loc='left', fontsize=17, color=INK, fontweight='bold', pad=44)
    ax.text(0, 1.012, f'{len(papers)} countries, {len(cross)} cross-country links. Lines connect countries whose authors co-wrote at least one paper.\n'
            f'Same-country collaboration (e.g. US-US, {dom_us} papers) is not drawn. Same scales on every map in this series.',
            transform=ax.transAxes, fontsize=11, color=MUTED)
    n_loc = B[(B.year >= y0) & (B.year <= y1)].row_id.nunique()
    fig.text(0.01, 0.01, f'Source: DGRL v2.1 dg.o corpus, author affiliations from OpenAlex, Crossref, ROR and manual lookups ({n_loc} papers with author-level locations).',
             fontsize=8.5, color=MUTED)
    plt.savefig(out, facecolor=BG); plt.close(fig)
    print(f'saved {os.path.basename(out)}: {len(papers)} countries, {len(cross)} cross-country links, {n_loc} papers')

for y0, y1, name in [(2000, 2005, 'country_collaboration_map_2000_2005.png'), (2000, 2012, 'country_collaboration_map_2000_2012.png'),
                     (2000, 2024, 'country_collaboration_map.png')]:
    draw(y0, y1, os.path.join(OUT, name))

# ---------- rankings: which countries collaborate across borders most, and most widely ----------
RANGES = [(2000, 2005), (2006, 2012), (2013, 2024), (2000, 2012), (2000, 2024)]
rows, pair_rows = [], []
for y0, y1 in RANGES:
    papers, wts, topics = network(y0, y1)
    S = B[(B.year >= y0) & (B.year <= y1)]
    cc_of = S.groupby('row_id').country.agg(lambda s: set(s))
    for c in papers.index:
        mine = cc_of[cc_of.map(lambda s: c in s)]
        intl = int((mine.map(len) > 1).sum())
        partners = {v if u == c else u for (u, v) in wts if c in (u, v) and u != v}
        rows.append({'range': f'{y0}-{y1}', 'country': c, 'region': REGION_OF.get(c, ''), 'papers': int(papers[c]),
                     'partner_countries': len(partners), 'international_papers': intl,
                     'international_share_%': round(100 * intl / papers[c], 1), 'domestic_papers': wts.get((c, c), 0)})
    for (u, v), w in wts.items():
        if u != v:
            pair_rows.append({'range': f'{y0}-{y1}', 'country_a': u, 'country_b': v, 'papers': w, 'topics': len(topics[(u, v)])})
rank = pd.DataFrame(rows).sort_values(['range', 'partner_countries', 'international_papers'], ascending=[True, False, False])
pairs = pd.DataFrame(pair_rows).sort_values(['range', 'papers', 'topics'], ascending=[True, False, False])
rank.to_csv(os.path.join(OUT, 'country_collaboration_rankings.csv'), index=False)
pairs.to_csv(os.path.join(OUT, 'country_pair_rankings.csv'), index=False)
full = rank[rank.range == '2000-2024']
print('\nMost partner countries, 2000 to 2024:\n', full.head(10)[['country', 'partner_countries', 'international_papers', 'papers', 'international_share_%']].to_string(index=False))
print('\nMost internationally co-authored papers, 2000 to 2024:\n', full.sort_values('international_papers', ascending=False).head(10)[['country', 'international_papers', 'partner_countries', 'papers']].to_string(index=False))
print('\nStrongest country pairs, 2000 to 2024:\n', pairs[pairs.range == '2000-2024'].head(10).to_string(index=False))


## Step 15. Interactive collaboration atlas
Builds one self-contained HTML page from the paper records (`authorships.csv`, `dgrl_master.csv`). Before writing, it checks that the records reproduce every link and weight in `country_network.gexf`, and that the page's headline numbers match `key_numbers_by_phase.csv`, so the page can never disagree with the other outputs.

The page has three tabs: a map (click a country for its partners, topics, papers per year and top institutions, or a link for its joint papers), topics over time (topics by year, author regions by year, topics by region) and the paper list behind any filter. Filters for years (cumulative: 2000 to 2005, 2000 to 2012, 2000 to 2024, or any year on the slider), topic, country and author apply to every tab, and a coverage note shows how many papers in view have no author location.

Writes `gephi/country_collaboration_map.html`.

In [ ]:
import os, json, itertools, pandas as pd, networkx as nx
GEO = os.path.join(REV, 'geo', 'ne_110m_admin_0_countries.geojson')   # downloaded in Step 14
G = nx.read_gexf(os.path.join(OUT, 'country_network.gexf')); W = json.load(open(GEO))
A = pd.read_csv(os.path.join(REV, 'authorships.csv'), keep_default_na=False)
# links found for papers without a DOI (Step 19 check, exact title matches), used only where the DGRL record has no DOI
_xl = os.path.join(IN, 'paper_links_extra.csv')
EXTRA_LINK = dict(zip(*pd.read_csv(_xl, keep_default_na=False)[['row_id', 'link']].T.values)) if os.path.exists(_xl) else {}
m = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'), keep_default_na=False)

pt, names, feats = {}, {}, []
for f in W['features']:
    pr = f['properties']
    if pr.get('NAME') == 'Antarctica': continue
    iso = pr.get('ISO_A2_EH') or pr.get('ISO_A2')
    if iso and iso != '-99':
        pt[iso] = [round(pr['LABEL_X'], 2), round(pr['LABEL_Y'], 2)]; names[iso] = pr.get('NAME_EN') or pr['NAME']
    g = f['geometry']
    polys = g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]
    keep = []
    for poly in polys:
        ring, prev = [], None
        for x, y in poly[0]:
            q = (round(x, 1), round(y, 1))
            if q != prev: ring.append(list(q)); prev = q
        if len(ring) > 3: keep.append([ring])
    feats.append({'type': 'Feature', 'properties': {'iso': iso}, 'geometry': {'type': 'MultiPolygon', 'coordinates': keep}})
EXTRA = {'SG': (103.8, 1.35, 'Singapore'), 'HK': (114.17, 22.32, 'Hong Kong'), 'BN': (114.7, 4.5, 'Brunei'),
         'LU': (6.13, 49.6, 'Luxembourg'), 'MO': (113.55, 22.2, 'Macao'), 'SI': (14.8, 46.1, 'Slovenia'), 'MT': (14.4, 35.9, 'Malta')}
for c, (x, y, n) in EXTRA.items():
    pt.setdefault(c, [x, y]); names.setdefault(c, n)
SHORT = {'US': 'United States', 'CN': 'China', 'KR': 'South Korea', 'RU': 'Russia', 'GB': 'United Kingdom'}
names.update(SHORT)
assert all(n in pt for n in G.nodes), [n for n in G.nodes if n not in pt]
nodes = [{'id': n, 'name': names[n], 'lon': pt[n][0], 'lat': pt[n][1], 'first': int(a['first_year']), 'region': a['region']}
         for n, a in G.nodes(data=True)]
codes = [n['id'] for n in nodes]
paper_cc = sorted({c for s in m.countries_all for c in s.split('|') if c} - set(codes))
codes += paper_cc
assert all(c in names for c in codes), [c for c in codes if c not in names]
cidx = {c: i for i, c in enumerate(codes)}
REGIONS = ['North America', 'Europe', 'Latin America', 'Asia', 'Oceania', 'Africa']
ridx = {r: i for i, r in enumerate(REGIONS)}
TOPICS = m.drop_duplicates('topic_k10').sort_values('topic_k10').topic_name_k10.tolist()

B = A[A.country != '']
aidx = {k: i for i, k in enumerate(sorted(A.author_key.unique()))}
author_names = A.drop_duplicates('author_key').set_index('author_key').author_name.reindex(sorted(A.author_key.unique())).tolist()
insts = sorted({x for x in B.institution if x})
iidx = {x: i for i, x in enumerate(insts)}
au_by = {r: [[aidx[k], cidx[c], iidx.get(n, -1)] for k, c, n in zip(g.author_key, g.country, g.institution)] for r, g in B.groupby('row_id')}
ak_by = {r: [aidx[k] for k in g.sort_values('order').author_key] for r, g in A.groupby('row_id')}
names_by = {r: ', '.join(g.sort_values('order').author_name) for r, g in A.groupby('row_id')}
papers = []
for r in m.sort_values('row_id').itertuples():
    pc = [cidx[c] for c in r.countries_all.split('|') if c]
    prg = [ridx[x] for x in r.regions_all.split('|') if x]
    papers.append([int(r.year), int(r.topic_k10), r.title, r.doi or EXTRA_LINK.get(r.row_id, ''), names_by.get(r.row_id, r.authors), pc, prg, au_by.get(r.row_id, []), ak_by.get(r.row_id, [])])

# the page rebuilds the network from these records: they must reproduce the Gephi file exactly
Wt, Pr = {}, {}
for r, g in B.groupby('row_id'):
    rows, seen = list(zip(g.author_key, g.country)), set()
    for (k1, c1), (k2, c2) in itertools.combinations(rows, 2):
        if k1 == k2: continue
        e = tuple(sorted((c1, c2))); Pr[e] = Pr.get(e, 0) + 1; seen.add(e)
    for e in seen: Wt[e] = Wt.get(e, 0) + 1
gx = {tuple(sorted((u, v))): (int(d['weight']), int(d['author_pairs'])) for u, v, d in G.edges(data=True)}
assert gx == {e: (Wt[e], Pr[e]) for e in Wt}
DATA = {'land': {'type': 'FeatureCollection', 'features': feats}, 'nodes': nodes,
        'codes': codes, 'names': [names[c] for c in codes], 'regions': REGIONS, 'topics': TOPICS, 'papers': papers,
        'authors': author_names, 'institutions': insts}
print(f"{len(papers)} papers; {sum(1 for p in papers if p[7])} with author-level locations; network matches country_network.gexf")

# The page's findings strip must match key_numbers_by_phase.csv (Step 8)
kn = pd.read_csv(os.path.join(REV, 'key_numbers_by_phase.csv'))
for (a, b_, _), row in zip(PHASES, kn.itertuples()):
    ps = [p for p in papers if a <= p[0] <= b_]; loc = [p for p in ps if p[5]]
    assert len(ps) == row.papers and len(loc) == row.with_country
    assert round(100 * sum(len(p[5]) > 1 for p in loc) / len(loc), 1) == row._6   # intl_coauthored_%
print('Findings strip numbers match key_numbers_by_phase.csv')

TEMPLATE = r'''<title>dg.o Collaboration Atlas</title>
<meta name="claude:viewport-fit-sides" content="cover">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Space+Grotesk:wght@500;600&display=swap">
<style>
/* Layout: title, findings strip, filters, tabs (map | topics over time | papers), notes */
:root {
  --bg: #f7f7f4; --surface: #ffffff; --fg: #1f1f1e; --muted: #64635d; --line: #dedcd4;
  --land: #e4e3dc; --land-edge: #f7f7f4; --link: #3d3d3a; --hi: #c2410c; --ring: #ffffff;
  --chip: #efeee8; --chip-on: #1f1f1e; --chip-on-fg: #ffffff; --heat-0: #e8f0fb; --heat-1: #12407a;
  /* topic colors: 8 categorical slots in fixed order, two neutrals for the smallest-trend topics */
  --t0: #2a78d6; --t1: #eb6834; --t2: #1baf7a; --t3: #eda100; --t4: #e87ba4; --t5: #008300; --t6: #6250d6; --t7: #8a8984; --t8: #e34948; --t9: #c9c8c1;
  --display: "Space Grotesk", "Inter", system-ui, sans-serif;
  --body: "Inter", system-ui, -apple-system, "Segoe UI", sans-serif;
}
@media (prefers-color-scheme: dark) { :root:not([data-theme="light"]) {
  --bg: #161615; --surface: #1f1f1d; --fg: #ecebe6; --muted: #a3a29b; --line: #34332f;
  --land: #2e2e2b; --land-edge: #161615; --link: #d6d5cf; --hi: #fb923c; --ring: #161615;
  --chip: #2a2a27; --chip-on: #ecebe6; --chip-on-fg: #161615; --heat-0: #1d2a3b; --heat-1: #8cc0ff;
  --t0: #3987e5; --t1: #d95926; --t2: #199e70; --t3: #c98500; --t4: #d55181; --t5: #0ca30c; --t6: #9085e9; --t7: #8f8e88; --t8: #e66767; --t9: #55544f; color-scheme: dark } }
:root[data-theme="dark"] {
  --bg: #161615; --surface: #1f1f1d; --fg: #ecebe6; --muted: #a3a29b; --line: #34332f;
  --land: #2e2e2b; --land-edge: #161615; --link: #d6d5cf; --hi: #fb923c; --ring: #161615;
  --chip: #2a2a27; --chip-on: #ecebe6; --chip-on-fg: #161615; --heat-0: #1d2a3b; --heat-1: #8cc0ff;
  --t0: #3987e5; --t1: #d95926; --t2: #199e70; --t3: #c98500; --t4: #d55181; --t5: #0ca30c; --t6: #9085e9; --t7: #8f8e88; --t8: #e66767; --t9: #55544f; color-scheme: dark }
body { background: var(--bg); color: var(--fg); font-family: var(--body); line-height: 1.45; }
.wrap { max-width: 1320px; margin: 0 auto; padding-block: 24px 40px;
  padding-left: max(16px, env(safe-area-inset-left, 0px)); padding-right: max(16px, env(safe-area-inset-right, 0px)); }
h1 { font-family: var(--display); font-weight: 600; font-size: clamp(1.4rem, 2.6vw, 2rem); margin: 0 0 6px; text-wrap: balance; letter-spacing: -0.01em; }
h2 { font-family: var(--display); font-weight: 600; font-size: 1.1rem; margin: 0 0 2px; }
.lede { color: var(--muted); margin: 0 0 16px; max-width: 80ch; }
.label { font-size: 11px; text-transform: uppercase; letter-spacing: 0.08em; color: var(--muted); }
button:focus-visible, input:focus-visible, select:focus-visible, a:focus-visible { outline: 2px solid var(--hi); outline-offset: 2px; }

/* findings strip */
.strip { display: grid; grid-template-columns: repeat(5, minmax(0, 1fr)); gap: 1px; background: var(--line); border: 1px solid var(--line); border-radius: 12px; overflow: hidden; margin-bottom: 14px; }
@media (max-width: 900px) { .strip { grid-template-columns: repeat(2, minmax(0, 1fr)); } .strip .wide { grid-column: 1 / -1; } }
.tile { background: var(--surface); padding: 12px 14px; min-width: 0; }
.tile b { display: block; font-family: var(--display); font-size: 1.5rem; font-weight: 600; font-variant-numeric: tabular-nums; line-height: 1.2; }
.tile span { font-size: 12px; color: var(--muted); }
.rbar { display: flex; gap: 2px; height: 12px; margin: 8px 0 6px; border-radius: 3px; overflow: hidden; }
.rbar i { display: block; height: 100%; }
.rkey { display: flex; flex-wrap: wrap; gap: 2px 10px; font-size: 11px; color: var(--muted); font-variant-numeric: tabular-nums; }

/* filters */
.controls { display: flex; flex-wrap: wrap; gap: 10px 20px; align-items: center; margin-bottom: 10px; }
.ctl { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; min-width: 0; }
.seg { display: flex; flex-wrap: wrap; gap: 4px; }
.seg button, .zoom button, .btn { font: 500 13px var(--body); border: 0; border-radius: 999px; padding: 7px 12px; background: var(--chip); color: var(--fg); cursor: pointer; }
.seg button[aria-pressed="true"] { background: var(--chip-on); color: var(--chip-on-fg); }
select, input[type="search"] { font: 13px var(--body); padding: 6px 8px; border-radius: 8px; border: 1px solid var(--line); background: var(--surface); color: var(--fg); max-width: min(300px, 100%); }
.slider { display: flex; align-items: center; gap: 8px; }
.slider input { width: 180px; accent-color: var(--hi); }
.slider output { font-variant-numeric: tabular-nums; font-size: 13px; min-width: 9ch; }
.chips { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; min-height: 30px; margin-bottom: 12px; font-size: 12px; }
.chips .c { display: inline-flex; align-items: center; gap: 6px; background: var(--chip); border-radius: 999px; padding: 4px 6px 4px 10px; }
.chips .c button { border: 0; background: none; color: var(--muted); cursor: pointer; font-size: 14px; line-height: 1; padding: 0 2px; }
.chips .clear { border: 0; background: none; color: var(--hi); font: 500 12px var(--body); cursor: pointer; }

/* tabs */
.tabs { display: flex; gap: 2px; border-bottom: 1px solid var(--line); margin-bottom: 14px; overflow-x: auto; }
.tabs button { font: 500 14px var(--body); background: none; border: 0; padding: 10px 14px; color: var(--muted); border-bottom: 2px solid transparent; margin-bottom: -1px; white-space: nowrap; cursor: pointer; }
.tabs button:hover { color: var(--fg); }
.tabs button[aria-selected="true"] { color: var(--fg); border-bottom-color: var(--hi); }
.tabs button .n { font-variant-numeric: tabular-nums; color: var(--muted); font-weight: 400; margin-left: 4px; }

/* map */
.stage { display: grid; grid-template-columns: minmax(0, 1fr) 310px; gap: 16px; align-items: start; }
@media (max-width: 880px) { .stage { grid-template-columns: minmax(0, 1fr); } }
.card { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; min-width: 0; }
.mapbox { position: relative; overflow: hidden; }
svg#map { display: block; width: 100%; height: auto; touch-action: none; cursor: grab; }
.zoom { position: absolute; right: 10px; top: 10px; display: flex; flex-direction: column; gap: 4px; }
.zoom button { width: 34px; height: 34px; padding: 0; font-size: 16px; background: var(--surface); border: 1px solid var(--line); }
.hint { position: absolute; left: 12px; top: 10px; font-size: 12px; color: var(--muted); background: color-mix(in srgb, var(--surface) 85%, transparent); padding: 2px 6px; border-radius: 6px; }
@media (max-width: 600px) { .hint { display: none; } }
.legend { display: flex; flex-wrap: wrap; gap: 8px 22px; padding: 12px 14px; border-top: 1px solid var(--line); font-size: 12px; color: var(--muted); align-items: center; }
.legend .grp { display: flex; flex-wrap: wrap; gap: 6px 12px; align-items: center; }
.sw { display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 5px; vertical-align: -1px; }
.sq { display: inline-block; width: 10px; height: 10px; border-radius: 2px; margin-right: 5px; vertical-align: -1px; }
.panel { padding: 16px; }
.panel .sub { color: var(--muted); font-size: 12px; margin-bottom: 12px; }
.stats { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-bottom: 14px; }
.stat b { display: block; font-family: var(--display); font-size: 1.3rem; font-variant-numeric: tabular-nums; font-weight: 600; }
.stat span { font-size: 12px; color: var(--muted); }
.plist { list-style: none; margin: 4px 0 0; padding: 0; max-height: 220px; overflow-y: auto; font-size: 13px; }
.plist li { display: grid; grid-template-columns: 1fr auto; gap: 8px; padding: 5px 0; border-bottom: 1px solid var(--line); cursor: pointer; }
.plist li:hover { color: var(--hi); }
.plist .n { font-variant-numeric: tabular-nums; color: var(--muted); }
.rk button { font-size: 12px; padding: 5px 9px; }
.btn.save { font-size: 12px; padding: 5px 11px; border: 1px solid var(--line); background: var(--surface); }
.btn.save:hover { border-color: var(--fg); }
.legend .btn.save { margin-left: auto; }
.toast { position: fixed; left: 50%; bottom: calc(20px + env(safe-area-inset-bottom, 0px)); transform: translateX(-50%); background: var(--chip-on); color: var(--chip-on-fg); padding: 8px 14px; border-radius: 8px; font-size: 13px; z-index: 30; max-width: calc(100% - 32px); }
.panel .go { margin: 12px 0 2px; width: 100%; background: var(--chip-on); color: var(--chip-on-fg); }
.panel .clear { margin-top: 8px; font: 500 12px var(--body); background: none; border: 0; color: var(--hi); cursor: pointer; padding: 0; }
.tbars { list-style: none; margin: 4px 0 0; padding: 0; display: grid; gap: 3px; }
.tb { width: 100%; display: grid; grid-template-columns: minmax(0, 1fr) auto; gap: 2px 8px; background: none; border: 0; padding: 3px 0; text-align: left; font: 12px var(--body); color: var(--fg); cursor: pointer; }
.tb .tv { color: var(--muted); font-variant-numeric: tabular-nums; }
.tb .bar { grid-column: 1 / -1; height: 6px; background: var(--chip); border-radius: 3px; overflow: hidden; }
.tb .bar span { display: block; height: 100%; background: var(--link); opacity: 0.55; border-radius: 3px; }
.tb:hover .tn, .tb.on .tn { color: var(--hi); }
.tb.on .bar span { background: var(--hi); opacity: 1; }
.tip { position: fixed; pointer-events: none; background: var(--surface); color: var(--fg); border: 1px solid var(--line); border-radius: 8px; padding: 8px 10px; font-size: 12px; line-height: 1.4; box-shadow: 0 4px 14px rgb(0 0 0 / 0.15); max-width: 260px; z-index: 20; }
.tip b { font-size: 13px; }
.tip .m { color: var(--muted); }
.land { fill: var(--land); stroke: var(--land-edge); stroke-width: 0.5; vector-effect: non-scaling-stroke; }
.arc { fill: none; stroke: var(--link); stroke-linecap: round; vector-effect: non-scaling-stroke; cursor: pointer; }
.arc.hi { stroke: var(--hi); }
.node { stroke: var(--ring); vector-effect: non-scaling-stroke; cursor: pointer; }
.lbl { font: 600 11px var(--body); fill: var(--fg); paint-order: stroke; stroke: var(--surface); stroke-width: 3px; stroke-linejoin: round; pointer-events: none; }

/* charts */
.chart { padding: 16px; margin-bottom: 16px; }
.chart .head { display: flex; flex-wrap: wrap; justify-content: space-between; gap: 8px 16px; align-items: flex-start; margin-bottom: 10px; }
.chart .head p { margin: 2px 0 0; color: var(--muted); font-size: 12px; max-width: 75ch; }
.scroll { overflow-x: auto; }
.scroll svg { display: block; min-width: 720px; width: 100%; height: auto; }
.ax { font: 11px var(--body); fill: var(--muted); font-variant-numeric: tabular-nums; }
.rowlbl { font: 12px var(--body); fill: var(--fg); cursor: pointer; }
.rowlbl.on { fill: var(--hi); font-weight: 600; }
.cell { cursor: pointer; }
.tk { font: 12px var(--body); color: var(--fg); background: none; border: 0; padding: 2px 4px; border-radius: 6px; cursor: pointer; }
.tk:hover, .tk.on { background: var(--chip); }
.tk.off { color: var(--muted); }
.tk.off .sq { opacity: 0.35; }
.cell:hover { stroke: var(--fg); stroke-width: 1.5; }
.grid { stroke: var(--line); }

/* papers */
.ptools { display: flex; flex-wrap: wrap; gap: 10px 16px; align-items: center; padding: 14px 16px; border-bottom: 1px solid var(--line); }
.ptools input { width: min(360px, 100%); }
.ptools .cnt { color: var(--muted); font-size: 13px; }
.tw { overflow-x: auto; }
table { border-collapse: collapse; width: 100%; font-size: 13px; }
th, td { text-align: left; padding: 8px 12px 8px 0; border-bottom: 1px solid var(--line); vertical-align: top; }
th:first-child, td:first-child { padding-left: 16px; }
th { font-size: 11px; text-transform: uppercase; letter-spacing: 0.06em; color: var(--muted); font-weight: 600; white-space: nowrap; }
td.y { font-variant-numeric: tabular-nums; white-space: nowrap; }
td.t { min-width: 280px; max-width: 520px; }
td.t a { color: var(--fg); text-decoration-color: var(--line); text-underline-offset: 2px; }
td.t a:hover { color: var(--hi); }
td .au { color: var(--muted); font-size: 12px; margin-top: 2px; }
td.c { min-width: 140px; }
td .pl { color: var(--muted); font-style: italic; }
td.tp { min-width: 160px; color: var(--muted); }
.more { padding: 12px 16px; }
.empty { padding: 24px 16px; color: var(--muted); }
.cov { font-size: 12px; color: var(--muted); background: var(--chip); border-radius: 8px; padding: 8px 10px; margin: 0 0 12px; }
.cov b { color: var(--fg); font-weight: 600; }
.plist li.ni { cursor: default; }
.plist li.ni:hover { color: inherit; }
#author { width: 200px; }
.note { color: var(--muted); font-size: 12px; margin-top: 18px; max-width: 95ch; }
.note.about summary { cursor: pointer; color: var(--ink, inherit); } .note.about ul { margin: 6px 0 0 18px; padding: 0; } .note.about li { margin: 3px 0; }
@media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
</style>

<div class="wrap">
  <h1>Who writes with whom at dg.o, 2000 to 2024</h1>
  <p class="lede">Co-authorship across countries in 812 dg.o papers. Filter by period, topic or year, click a country or a link on the map, and every tab and number below updates to match.</p>

  <section class="strip" id="strip" aria-label="Key numbers for the current filters"></section>

  <div class="controls">
    <div class="ctl"><span class="label">Years</span>
      <div class="seg" id="phase" role="group" aria-label="Period">
        <button type="button" data-y="2005" aria-pressed="false" title="NSF PI meetings era">2000 to 2005</button>
        <button type="button" data-y="2012" aria-pressed="false" title="Through the DGS of North America era">2000 to 2012</button>
        <button type="button" data-y="2024" aria-pressed="true" title="All years">2000 to 2024</button>
      </div>
    </div>
    <div class="ctl"><label class="label" for="topic">Topic</label><select id="topic"><option value="-1">All topics</option></select></div>
    <div class="ctl"><label class="label" for="pick">Country</label><select id="pick"><option value="">All countries</option></select></div>
    <div class="ctl"><label class="label" for="author">Author</label><input type="search" id="author" list="authorList" placeholder="Type a name" autocomplete="off"><datalist id="authorList"></datalist></div>
    <div class="ctl slider"><label class="label" for="yr">Through</label>
      <input type="range" id="yr" min="2000" max="2024" step="1" value="2024"><output id="yrOut" for="yr">2024</output>
      <button type="button" class="btn" id="play" aria-label="Play years from 2000 to 2024">▶ Play</button></div>
  </div>
  <div class="chips" id="chips" aria-live="polite"></div>

  <nav class="tabs" role="tablist">
    <button type="button" role="tab" data-tab="map" id="tab-map" aria-selected="true">Map</button>
    <button type="button" role="tab" data-tab="topics" id="tab-topics" aria-selected="false">Topics over time</button>
    <button type="button" role="tab" data-tab="papers" id="tab-papers" aria-selected="false">Papers <span class="n" id="pcount"></span></button>
  </nav>

  <section id="view-map">
    <div class="stage">
      <div class="card mapbox">
        <svg id="map" role="img" aria-label="World map of co-authorship links between countries"></svg>
        <div class="hint">Scroll or pinch to zoom · drag to pan · click a country or link</div>
        <div class="zoom"><button type="button" id="zin" aria-label="Zoom in">+</button><button type="button" id="zout" aria-label="Zoom out">−</button><button type="button" id="zreset" aria-label="Reset view">⟲</button></div>
        <div class="legend"><div class="grp regions"></div>
          <div class="grp">Circle size = papers with an author there · Line width = papers co-written across the two countries</div>
          <button type="button" class="btn save" data-save="map" title="Save the current map view as a high-resolution PNG ">Save PNG</button></div>
      </div>
      <aside class="card panel" id="panel" aria-live="polite"></aside>
    </div>
  </section>

  <section id="view-topics" hidden>
    <div class="card chart">
      <div class="head"><div><h2>Topics by year</h2><p id="heatSub"></p></div>
        <div class="ctl"><button type="button" class="btn save" data-save="heat" title="Save this chart as a high-resolution PNG ">Save PNG</button><div class="seg" id="heatView" role="group" aria-label="Chart type">
          <button type="button" data-v="bars" aria-pressed="true">Bars</button>
          <button type="button" data-v="heat" aria-pressed="false">Heatmap</button></div>
        <div class="seg" id="heatMode" role="group" aria-label="Values">
          <button type="button" data-m="share" aria-pressed="true">Share of year</button>
          <button type="button" data-m="count" aria-pressed="false">Number of papers</button></div></div></div>
      <div class="scroll"><svg id="heat" role="img" aria-label="Topics by year"></svg></div>
      <div class="legend" style="border:0;padding:8px 0 0" id="heatKey"></div>
    </div>
    <div class="card chart">
      <div class="head"><div><h2>Where authors were based, by year</h2><p id="regSub"></p></div>
        <button type="button" class="btn save" data-save="reg" title="Save this chart as a high-resolution PNG ">Save PNG</button></div>
      <div class="scroll"><svg id="reg" role="img" aria-label="Stacked bars of author regions by year"></svg></div>
      <div class="legend" style="border:0;padding:8px 0 0"><div class="grp regions sqk"></div></div>
    </div>
    <div class="card chart">
      <div class="head"><div><h2>Topics by region</h2><p id="trSub"></p></div>
        <button type="button" class="btn save" data-save="tr" title="Save this chart as a high-resolution PNG ">Save PNG</button></div>
      <div class="scroll"><svg id="tr" role="img" aria-label="Heatmap of topics by author region"></svg></div>
      <div class="legend" style="border:0;padding:8px 0 0" id="trKey"></div>
    </div>
  </section>

  <section id="view-papers" hidden>
    <div class="card">
      <div class="ptools"><input type="search" id="q" placeholder="Search titles and authors" aria-label="Search titles and authors"><span class="cnt" id="pinfo"></span></div>
      <div class="tw"><table><thead><tr><th>Year</th><th>Paper</th><th>Author countries</th><th>Topic</th></tr></thead><tbody id="ptable"></tbody></table></div>
      <div class="more" id="pmore"></div>
    </div>
  </section>

  <p class="note">Source: DGRL v2.1 dg.o corpus (__N_PAPER__ papers); affiliations from OpenAlex, Crossref, ROR and manual lookups; topics are the k=10 BERTopic model (Table 2).</p>
  <details class="note about"><summary>How the numbers are counted</summary><ul><li>The map, country filter and Papers tab use author-level locations (__N_AUTH__ papers): a paper counts once for each country among its authors, and a line means authors in two countries co-wrote at least one paper.</li><li>The findings strip and region chart use paper-level affiliations (__N_PAPER__ papers); a paper with authors in several regions is split evenly between them.</li><li>Same-country collaboration appears in the side panel and tooltips, not as lines. Circle sizes use one scale across all views. Countries in italics in the Papers tab are paper-level only.</li></ul></details>
  <div class="tip" id="tip" hidden></div>
  <div class="toast" id="toast" role="status" hidden></div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>
<script>
const DATA = __DATA__;
const REGION = { "North America": "#2a78d6", "Europe": "#eb6834", "Latin America": "#1baf7a", "Asia": "#eda100", "Oceania": "#e87ba4", "Africa": "#008300" };
const PHASES = ["2000 to 2005", "2006 to 2012", "2013 to 2024"];
const TOPICS = DATA.topics, REGIONS = DATA.regions, CODES = DATA.codes, NAMES = DATA.names;
const YEARS = d3.range(2000, 2025);
const phaseOf = y => y <= 2005 ? 0 : y <= 2012 ? 1 : 2;
const nodes = DATA.nodes, byId = Object.fromEntries(nodes.map(n => [n.id, n]));
const NC = nodes.length;  // the first NC codes are map nodes
const P = DATA.papers.map(([y, t, title, doi, authors, pc, pr, au, ak], i) => ({ i, y, t, title, doi, authors, pc, pr, au, ak,
  ac: [...new Set(au.map(a => a[1]))] }));
const AUTHORS = DATA.authors, INSTS = DATA.institutions;

const st = { phase: -1, topic: -1, yearMax: 2024, yearExact: null, country: null, link: null, author: null, tab: "map", rank: "partners", heat: "share", hview: "bars", q: "", shown: 100 };
let S = null, k = 1, playing = null;

// ---------- filters ----------
const inTime = p => (st.phase < 0 || phaseOf(p.y) === st.phase) && (st.yearExact != null ? p.y === st.yearExact : p.y <= st.yearMax);
const inTopic = p => st.topic < 0 || p.t === st.topic;
const inAuthor = p => !st.author || p.ak.some(a => st.author.ids.includes(a));
const ci = id => CODES.indexOf(id);
const inPlace = p => {
  if (st.link) { const a = ci(st.link[0]), b = ci(st.link[1]); return linkPapers(p, a, b); }
  if (st.country) return p.ac.includes(ci(st.country));
  return true;
};
function linkPapers(p, a, b) {
  const au = p.au;
  for (let i = 0; i < au.length; i++) for (let j = i + 1; j < au.length; j++) {
    if (au[i][0] === au[j][0]) continue;
    const x = au[i][1], y = au[j][1];
    if ((x === a && y === b) || (x === b && y === a)) return true;
  }
  return false;
}
function timeLabel() {
  if (st.yearExact != null) return String(st.yearExact);
  if (st.phase >= 0) { const [a, b] = [[2000, 2005], [2006, 2012], [2013, 2024]][st.phase]; return st.yearMax < b ? `${a} to ${Math.max(a, st.yearMax)}` : PHASES[st.phase]; }
  return `2000 to ${st.yearMax}`;
}

// ---------- network from paper records ----------
function compute() {
  const node = {}, edge = {};
  for (let c = 0; c < NC; c++) node[CODES[c]] = { papers: 0, authors: new Set(), dom: 0, intl: 0, topics: Array(TOPICS.length).fill(0), all: 0 };
  for (const p of P) {
    if (!inTime(p) || !inAuthor(p) || !p.au.length) continue;
    p.ac.forEach(c => { const n = node[CODES[c]]; n.all++; n.topics[p.t]++; });
    if (!inTopic(p)) continue;
    p.ac.forEach(c => { node[CODES[c]].papers++; if (p.ac.length > 1) node[CODES[c]].intl++; });
    p.au.forEach(a => node[CODES[a[1]]].authors.add(a[0]));
    const seen = new Set();
    for (let i = 0; i < p.au.length; i++) for (let j = i + 1; j < p.au.length; j++) {
      if (p.au[i][0] === p.au[j][0]) continue;
      const [s, t] = [CODES[p.au[i][1]], CODES[p.au[j][1]]].sort(), key = s + "|" + t;
      const e = edge[key] || (edge[key] = { s, t, w: 0, pairs: 0, first: 9999, last: 0, p: [0, 0, 0], tp: Array(TOPICS.length).fill(0) });
      e.pairs++;
      if (!seen.has(key)) { seen.add(key); e.w++; e.first = Math.min(e.first, p.y); e.last = Math.max(e.last, p.y); e.p[phaseOf(p.y)]++; e.tp[p.t]++; }
    }
  }
  Object.values(node).forEach(n => n.authors = n.authors.size);
  const edges = Object.values(edge);
  edges.filter(e => e.s === e.t).forEach(e => node[e.s].dom = e.w);
  return { node, cross: edges.filter(e => e.s !== e.t), same: edges.filter(e => e.s === e.t) };
}

// ---------- shared helpers ----------
const fmt = n => n.toLocaleString("en-US");
const pct = (a, b) => b ? Math.round(1000 * a / b) / 10 : 0;
const pl = (n, w) => `${fmt(n)} ${w}${n === 1 ? "" : "s"}`;
const tip = document.getElementById("tip");
function showTip(html, ev) {
  tip.innerHTML = html; tip.hidden = false;
  let x = ev.clientX + 14, y = ev.clientY + 14;
  if (x + tip.offsetWidth > innerWidth - 8) x = ev.clientX - tip.offsetWidth - 14;
  if (y + tip.offsetHeight > innerHeight - 8) y = ev.clientY - tip.offsetHeight - 14;
  tip.style.left = Math.max(8, x) + "px"; tip.style.top = Math.max(8, y) + "px";
}
const hideTip = () => { tip.hidden = true; };
const topTopics = (arr, n = 3) => arr.map((v, i) => [i, v]).filter(d => d[1] > 0).sort((a, b) => b[1] - a[1]).slice(0, n);
const esc = s => String(s).replace(/[&<>"]/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
document.querySelectorAll(".regions").forEach(el => el.innerHTML = REGIONS.map(r =>
  `<span><span class="${el.classList.contains("sqk") ? "sq" : "sw"}" style="background:${REGION[r]}"></span>${r}</span>`).join(""));

// ---------- findings strip (paper-level, matches key_numbers_by_phase.csv) ----------
function strip() {
  const ps = P.filter(p => inTime(p) && inTopic(p) && inAuthor(p)), loc = ps.filter(p => p.pc.length);
  const cc = new Set(); loc.forEach(p => p.pc.forEach(c => cc.add(c)));
  const intl = loc.filter(p => p.pc.length > 1).length;
  const frac = Array(REGIONS.length).fill(0);
  loc.forEach(p => { if (p.pr.length) p.pr.forEach(r => frac[r] += 1 / p.pr.length); });
  const tot = d3.sum(frac) || 1;
  const scope = [timeLabel(), st.topic < 0 ? null : TOPICS[st.topic]].filter(Boolean).join(" · ");
  document.getElementById("strip").innerHTML = `
    <div class="tile"><b>${fmt(ps.length)}</b><span>papers · ${esc(scope)}</span></div>
    <div class="tile"><b>${pct(loc.length, ps.length)}%</b><span>have author affiliations (${fmt(loc.length)})</span></div>
    <div class="tile"><b>${cc.size}</b><span>countries</span></div>
    <div class="tile"><b>${pct(intl, loc.length)}%</b><span>co-authored across countries</span></div>
    <div class="tile wide"><span>Share of authorship by region</span>
      <div class="rbar">${REGIONS.map((r, i) => frac[i] ? `<i style="width:${100 * frac[i] / tot}%;background:${REGION[r]}" title="${r} ${pct(frac[i], tot)}%"></i>` : "").join("")}</div>
      <div class="rkey">${REGIONS.map((r, i) => frac[i] ? `<span><span class="sq" style="background:${REGION[r]}"></span>${r.replace("North America", "N. America").replace("Latin America", "Latin Am.")} ${pct(frac[i], tot)}%</span>` : "").join("")}</div></div>`;
}

// ---------- filter chips ----------
function chips() {
  const c = [];
  if (st.phase >= 0) c.push(["Period: " + PHASES[st.phase], () => setPhase(-1)]);
  if (st.yearExact != null) c.push(["Year: " + st.yearExact, () => { st.yearExact = null; refresh(); }]);
  else if (st.yearMax < 2024) c.push(["2000 to " + st.yearMax, () => setYear(2024)]);
  if (st.topic >= 0) c.push(["Topic: " + TOPICS[st.topic], () => setTopic(-1)]);
  if (st.country) c.push(["Country: " + byId[st.country].name, () => select(null)]);
  if (st.link) c.push([`Link: ${byId[st.link[0]].name} and ${byId[st.link[1]].name}`, () => select(null)]);
  if (st.author) c.push(["Author: " + st.author.name, () => setAuthor(null)]);
  if (st.q) c.push([`Search: "${st.q}"`, () => { st.q = ""; document.getElementById("q").value = ""; refresh(); }]);
  const el = document.getElementById("chips");
  el.innerHTML = c.length ? c.map(([t], i) => `<span class="c">${esc(t)}<button type="button" data-i="${i}" aria-label="Remove filter ${esc(t)}">×</button></span>`).join("")
    + `<button type="button" class="clear" id="clearAll">Clear all</button>` : `<span class="label">No filters: showing all 812 papers</span>`;
  el.querySelectorAll("button[data-i]").forEach(b => b.onclick = () => c[+b.dataset.i][1]());
  const ca = document.getElementById("clearAll");
  if (ca) ca.onclick = () => { stopPlay(); Object.assign(st, { phase: -1, topic: -1, yearMax: 2024, yearExact: null, country: null, link: null, author: null, q: "" });
    document.getElementById("q").value = ""; document.getElementById("author").value = ""; document.getElementById("pick").value = ""; syncControls(); refresh(); svg.transition().duration(300).call(zoom.transform, d3.zoomIdentity); };
}

// ---------- map ----------
const W = 960, H = 500, WMAX_FIXED = 9;
const proj = d3.geoEqualEarth().fitExtent([[8, 8], [W - 8, H - 8]], { type: "Sphere" });
proj.scale(proj.scale() * 1.06).translate([W / 2, H / 2 + 22]);
nodes.forEach(n => { [n.x, n.y] = proj([n.lon, n.lat]); });
const svg = d3.select("#map").attr("viewBox", `0 0 ${W} ${H}`);
const g = svg.append("g");
g.append("g").selectAll("path").data(DATA.land.features).join("path").attr("class", "land").attr("d", d3.geoPath(proj));
const gArcs = g.append("g"), gNodes = g.append("g"), gLbl = g.append("g");
// one fixed circle scale for every view: the most papers any country has across all years
const PMAX = (() => { const c = {}; P.forEach(p => p.ac.forEach(x => c[x] = (c[x] || 0) + 1)); return d3.max(Object.values(c)); })();
const rScale = () => d3.scaleSqrt().domain([0, PMAX]).range([0, 26]);
const rad = id => Math.max(4, rScale()(S.node[id].papers));

function arcPath(e) {
  const a = byId[e.s], b = byId[e.t], dx = b.x - a.x, dy = b.y - a.y;
  return `M${a.x},${a.y}Q${(a.x + b.x) / 2 - dy * 0.18},${(a.y + b.y) / 2 + dx * 0.18} ${b.x},${b.y}`;
}
const partners = id => S.cross.filter(e => e.s === id || e.t === id).map(e => ({ id: e.s === id ? e.t : e.s, w: e.w, e }))
  .sort((a, b) => b.w - a.w || byId[a.id].name.localeCompare(byId[b.id].name));
const isLink = e => st.link && e.s === st.link[0] && e.t === st.link[1];

function nodeTip(n) {
  const s = S.node[n.id], tt = topTopics(s.topics, 2).map(([i, v]) => `${TOPICS[i]} (${v})`).join("<br>");
  return `<b>${n.name}</b> <span class="m">${n.region}</span><br>${pl(s.papers, "paper")}, ${pl(s.authors, "author")}<br>`
    + `${pl(partners(n.id).length, "partner country")}, ${pl(s.dom, "same-country co-authored paper")}`
    + (tt ? `<br><span class="m">Top topics:</span><br>${tt}` : "") + `<br><span class="m">First dg.o paper: ${n.first} · click for details</span>`;
}
function linkTip(e) {
  const tt = topTopics(e.tp, 3).map(([i, v]) => `${TOPICS[i]} (${v})`).join("<br>");
  return `<b>${byId[e.s].name} and ${byId[e.t].name}</b><br>${pl(e.w, "paper")}, ${pl(e.pairs, "author pair")}<br>`
    + `<span class="m">${e.first === e.last ? e.first : e.first + " to " + e.last} · by period ${e.p.join(" / ")}</span>`
    + (tt ? `<br><span class="m">Topics:</span><br>${tt}` : "") + `<br><span class="m">Click to see the papers</span>`;
}

// Greedy label placement for saved images: biggest first, try right, left, above, below; skip a label that still collides
function placeLabels(items, W0, H0) {
  const boxes = [], out = [], hit = (a, b) => a.x < b.x + b.w && b.x < a.x + a.w && a.y < b.y + b.h && b.y < a.y + a.h;
  items.forEach(it => boxes.push({ x: it.x - it.r, y: it.y - it.r, w: 2 * it.r, h: 2 * it.r }));
  items.forEach(it => {
    const w = it.text.length * it.fs * 0.58 + 2, h = it.fs + 2, g = 2;
    const tries = [[it.x + it.r + g, it.y - h / 2, "start"], [it.x - it.r - g - w, it.y - h / 2, "end"],
                   [it.x - w / 2, it.y - it.r - g - h, "middle"], [it.x - w / 2, it.y + it.r + g, "middle"]];
    for (const [bx, by, anchor] of tries) {
      const b = { x: bx, y: by, w, h };
      if (bx < 1 || by < 1 || bx + w > W0 - 1 || by + h > H0 - 1 || boxes.some(o => hit(b, o))) continue;
      boxes.push(b); out.push({ it, x: anchor === "start" ? bx : anchor === "end" ? bx + w : bx + w / 2, y: by + h - 3, anchor }); return;
    }
  });
  return out;
}
let EXPORT_LBL = false;   // set while saving the map PNG: full country names, Europe labelled in the inset instead
function drawMap() {
  const live = S.cross.slice().sort((a, b) => a.w - b.w);
  const focus = st.link ? new Set(st.link) : st.country ? new Set([st.country, ...partners(st.country).map(p => p.id)]) : null;
  const hiE = e => isLink(e) || (st.country && (e.s === st.country || e.t === st.country));
  gArcs.selectAll("path").data(live, e => e.s + e.t).join("path")
    .attr("class", e => "arc" + (hiE(e) ? " hi" : "")).attr("d", arcPath)
    .attr("stroke-width", e => 0.8 + 4.2 * Math.min(1, e.w / WMAX_FIXED) + (hiE(e) ? 0.8 : 0))
    .attr("stroke-opacity", e => focus ? (hiE(e) ? 0.95 : 0.07) : 0.3 + 0.55 * Math.min(1, e.w / WMAX_FIXED))
    .on("mousemove", (ev, e) => showTip(linkTip(e), ev)).on("mouseleave", hideTip)
    .on("click", (ev, e) => { ev.stopPropagation(); selectLink(e.s, e.t); });
  gArcs.selectAll("path.hi").raise();
  const order = [...nodes].sort((a, b) => S.node[b.id].papers - S.node[a.id].papers);
  const on = n => S.node[n.id].papers > 0;
  gNodes.selectAll("circle").data(order, n => n.id).join("circle")
    .attr("class", "node").attr("cx", n => n.x).attr("cy", n => n.y)
    .attr("r", n => rad(n.id) / Math.sqrt(k)).attr("stroke-width", 1.4)
    .attr("fill", n => REGION[n.region] || "#888")
    .attr("fill-opacity", n => focus ? (focus.has(n.id) ? 1 : 0.18) : on(n) ? 1 : 0.15)
    .on("mousemove", (ev, n) => showTip(nodeTip(n), ev)).on("mouseleave", hideTip)
    .on("click", (ev, n) => { ev.stopPropagation(); select(st.country === n.id ? null : n.id, false); });
  gNodes.selectAll("circle").filter(on).raise();
  const crowd = new Set(["BE", "CH", "AT", "DK", "IE", "PT", "LU", "SI", "MT", "HK", "MO"]);
  const top = order.filter(on).slice(0, 22);
  const lab = EXPORT_LBL ? order.filter(n => on(n) && (focus ? focus.has(n.id) : true) && n.region !== "Europe" && (focus || order.filter(on).indexOf(n) < 40))
    : order.filter(n => focus ? focus.has(n.id) : on(n) && (k >= 2.5 || (top.includes(n) && !crowd.has(n.id))));
  if (EXPORT_LBL) {
    const pl2 = placeLabels(lab.map(n => ({ id: n.id, x: n.x, y: n.y, r: rad(n.id), text: n.name, fs: 11 })), W, H);
    gLbl.selectAll("text").remove();
    gLbl.selectAll("text").data(pl2).join("text").attr("class", "lbl").attr("x", d => d.x).attr("y", d => d.y)
      .attr("text-anchor", d => d.anchor).style("font-size", "11px").style("stroke-width", "3px").text(d => d.it.text);
    return;
  }
  gLbl.selectAll("text").data(lab, n => n.it ? "x" + n.it.id : n.id).join("text").attr("class", "lbl").attr("text-anchor", null)
    .attr("x", n => n.x + rad(n.id) / Math.sqrt(k) + 3 / k).attr("y", n => n.y + 4 / k)
    .style("font-size", (11 / k) + "px").style("stroke-width", (3 / k) + "px").text(n => EXPORT_LBL || k >= 2.5 || focus ? n.name : n.id);
}

function topicBars(arr, total) {
  const max = d3.max(arr) || 1;
  return `<ul class="tbars">` + arr.map((v, i) => [i, v]).filter(d => d[1] > 0).sort((a, b) => b[1] - a[1]).map(([i, v]) =>
    `<li><button type="button" class="tb${i === st.topic ? " on" : ""}" data-t="${i}" title="Show only ${esc(TOPICS[i])}">
      <span class="tn">${TOPICS[i]}</span><span class="tv">${v} · ${Math.round(100 * v / total)}%</span>
      <span class="bar"><span style="width:${100 * v / max}%"></span></span></button></li>`).join("") + `</ul>`;
}
// papers per year for one country (author-level), all years shown, current range in full color
function trend(id) {
  const c = ci(id), cnt = YEARS.map(() => 0);
  P.forEach(p => { if (inTopic(p) && inAuthor(p) && p.ac.includes(c)) cnt[p.y - 2000]++; });
  const max = Math.max(1, d3.max(cnt)), w = 278, h = 64, bw = w / YEARS.length;
  const inR = y => (st.phase < 0 || phaseOf(y) === st.phase) && (st.yearExact != null ? y === st.yearExact : y <= st.yearMax);
  const peak = cnt.indexOf(d3.max(cnt));
  return `<svg viewBox="0 0 ${w} ${h + 16}" width="100%" role="img" aria-label="Papers per year">
    <line x1="0" x2="${w}" y1="${h}" y2="${h}" class="grid"/>
    ${cnt.map((v, j) => v ? `<rect x="${j * bw + 1}" width="${bw - 2}" y="${h - h * v / max}" height="${h * v / max}" rx="1.5" fill="${REGION[byId[id].region]}" opacity="${inR(2000 + j) ? 1 : 0.3}"><title>${2000 + j}: ${pl(v, "paper")}</title></rect>` : "").join("")}
    <text class="ax" x="0" y="${h + 13}">2000</text><text class="ax" x="${w}" y="${h + 13}" text-anchor="end">2024</text>
    ${cnt[peak] ? `<text class="ax" x="${Math.min(w - 40, Math.max(30, peak * bw + bw / 2))}" y="${h + 13}" text-anchor="middle">peak ${cnt[peak]} in ${2000 + peak}</text>` : ""}</svg>`;
}
// institutions in one country, counted once per paper, within the current filters
function instList(id) {
  const c = ci(id), m = new Map();
  P.forEach(p => {
    if (!inTime(p) || !inTopic(p) || !inAuthor(p)) return;
    new Set(p.au.filter(a => a[1] === c && a[2] >= 0).map(a => a[2])).forEach(i => m.set(i, (m.get(i) || 0) + 1));
  });
  const rows = [...m].sort((a, b) => b[1] - a[1] || INSTS[a[0]].localeCompare(INSTS[b[0]]));
  if (!rows.length) return `<p class="sub">No institution names recorded for these papers.</p>`;
  return `<ul class="plist">${rows.slice(0, 12).map(([i, v]) => `<li class="ni"><span>${esc(INSTS[i])}</span><span class="n">${v}</span></li>`).join("")}</ul>
    ${rows.length > 12 ? `<p class="sub" style="margin:6px 0 0">${rows.length} institutions in total</p>` : ""}`;
}
// how much of the current view the map can show
function coverage() {
  const ps = P.filter(p => inTime(p) && inTopic(p) && inAuthor(p)), loc = ps.filter(p => p.au.length).length, miss = ps.length - loc;
  return `<p class="cov"><b>${fmt(loc)} of ${fmt(ps.length)}</b> papers in this view have author locations and appear on the map.${miss ? ` ${fmt(miss)} (${Math.round(100 * miss / Math.max(1, ps.length))}%) have none yet, so they are left out of country counts.` : ""}</p>`;
}

function rankList(lead) {
  const deg = id => partners(id).length, s = id => S.node[id];
  const key = { papers: id => s(id).papers, partners: id => deg(id), intl: id => s(id).intl }[st.rank];
  return lead.filter(n => key(n.id) > 0).sort((a, b) => key(b.id) - key(a.id) || s(b.id).intl - s(a.id).intl || s(b.id).papers - s(a.id).papers).slice(0, 10)
    .map(n => [n, key(n.id), st.rank === "intl" ? `${Math.round(100 * s(n.id).intl / s(n.id).papers)}% of its papers` : st.rank === "partners" ? `${s(n.id).intl} intl papers` : ""]);
}
function panel(nPapers) {
  const el = document.getElementById("panel"), go = `<button type="button" class="btn go" id="goPapers">See the ${pl(nPapers, "paper")}</button>`;
  if (st.link) {
    const [a, b] = st.link, e = S.cross.find(x => x.s === a && x.t === b);
    el.innerHTML = `<h2>${byId[a].name} and ${byId[b].name}</h2><div class="sub">${timeLabel()}${st.topic >= 0 ? " · " + TOPICS[st.topic] : ""}</div>
      ${e ? `<div class="stats"><div class="stat"><b>${e.w}</b><span>papers together</span></div><div class="stat"><b>${e.pairs}</b><span>author pairs</span></div>
      <div class="stat"><b>${e.first === e.last ? e.first : e.first + " to " + e.last}</b><span>years active</span></div><div class="stat"><b>${e.p.join(" / ")}</b><span>papers by period</span></div></div>
      <div class="label">Topics of their joint papers</div>${topicBars(e.tp, e.w)}` : `<p class="sub">No joint papers with these filters.</p>`}
      ${go}<button type="button" class="clear" id="clr">Back to all countries</button>`;
  } else if (st.country) {
    const n = byId[st.country], s = S.node[st.country], p = partners(st.country);
    el.innerHTML = `<h2>${n.name}</h2><div class="sub">${n.region} · first dg.o paper ${n.first} · ${timeLabel()}${st.topic >= 0 ? " · " + TOPICS[st.topic] : ""}</div>
      <div class="stats"><div class="stat"><b>${s.papers}</b><span>papers</span></div><div class="stat"><b>${s.authors}</b><span>authors</span></div>
      <div class="stat"><b>${p.length}</b><span>partner countries</span></div><div class="stat"><b>${s.dom}</b><span>same-country co-authored papers</span></div></div>
      <div class="label">Papers per year${st.topic >= 0 ? " · " + TOPICS[st.topic] : ""}</div>${trend(st.country)}
      <div class="label" style="margin-top:12px">What ${n.name} writes about</div>${s.all ? topicBars(s.topics, s.all) : `<p class="sub">No papers in this period.</p>`}
      <div class="label" style="margin-top:12px">Top institutions</div>${instList(st.country)}
      <div class="label" style="margin-top:12px">Partners by shared papers</div>
      ${p.length ? `<ul class="plist">${p.map(q => `<li data-a="${st.country}" data-b="${q.id}"><span>${byId[q.id].name}</span><span class="n">${q.w}</span></li>`).join("")}</ul>` : `<p class="sub">No cross-country co-authored papers here.</p>`}
      ${go}<button type="button" class="clear" id="clr">Back to all countries</button>`;
  } else {
    const act = new Set(); S.cross.forEach(e => { act.add(e.s); act.add(e.t); });
    const strong = S.cross.slice().sort((a, b) => b.w - a.w).slice(0, 8);
    const lead = nodes.filter(n => S.node[n.id].papers > 0).sort((a, b) => S.node[b.id].papers - S.node[a.id].papers);
    el.innerHTML = `<h2>${timeLabel()}</h2><div class="sub">${st.topic < 0 ? "All topics" : TOPICS[st.topic]}. Click a country or link for details.</div>
      <div class="stats"><div class="stat"><b>${S.cross.length}</b><span>cross-country links</span></div><div class="stat"><b>${lead.length}</b><span>countries on the map</span></div>
      <div class="stat"><b>${d3.sum(S.cross, e => e.w)}</b><span>link papers across countries</span></div><div class="stat"><b>${d3.sum(S.same, e => e.w)}</b><span>link papers within one country</span></div></div>
      ${coverage()}
      <div class="seg rk" role="group" aria-label="Rank countries by">
        ${[["papers", "Papers"], ["partners", "Partner countries"], ["intl", "International papers"]].map(([v, l]) => `<button type="button" data-r="${v}" aria-pressed="${st.rank === v}">${l}</button>`).join("")}</div>
      <div class="label" style="margin-top:8px">${{ papers: "Most papers", partners: "Most partner countries (widest collaboration)", intl: "Most papers co-written with another country" }[st.rank]}</div>
      <ul class="plist">${rankList(lead).map(([n, v, sub]) => `<li data-id="${n.id}"><span>${n.name}${sub ? ` <span class="n">${sub}</span>` : ""}</span><span class="n">${v}</span></li>`).join("")}</ul>
      ${strong.length ? `<div class="label" style="margin-top:12px">Strongest links</div><ul class="plist">${strong.map(e => `<li data-a="${e.s}" data-b="${e.t}"><span>${byId[e.s].name} and ${byId[e.t].name}</span><span class="n">${e.w}</span></li>`).join("")}</ul>` : ""}`;
  }
  el.querySelectorAll("li[data-id]").forEach(li => li.onclick = () => select(li.dataset.id));
  el.querySelectorAll(".rk button").forEach(b => b.onclick = () => { st.rank = b.dataset.r; panel(nPapers); });
  el.querySelectorAll("li[data-a]").forEach(li => li.onclick = () => selectLink(li.dataset.a, li.dataset.b));
  el.querySelectorAll("button.tb").forEach(b => b.onclick = () => setTopic(+b.dataset.t === st.topic ? -1 : +b.dataset.t));
  const c = document.getElementById("clr"); if (c) c.onclick = () => select(null);
  const gp = document.getElementById("goPapers"); if (gp) gp.onclick = () => showTab("papers");
}
function flyTo(ids) {
  const pts = ids.map(i => byId[i]);
  const [x0, x1] = d3.extent(pts, p => p.x), [y0, y1] = d3.extent(pts, p => p.y);
  const s = Math.max(1, Math.min(6, 0.8 / Math.max((x1 - x0 + 40) / W, (y1 - y0 + 40) / H)));
  svg.transition().duration(500).call(zoom.transform, d3.zoomIdentity.translate(W / 2, H / 2).scale(s).translate(-(x0 + x1) / 2, -(y0 + y1) / 2));
}
function select(id, fly = true) {
  st.country = id; st.link = null; st.shown = 100; document.getElementById("pick").value = id || ""; refresh();
  if (id && fly && st.tab === "map") flyTo([id, ...partners(id).map(p => p.id)]);
}
function selectLink(a, b) { [a, b] = [a, b].sort(); st.link = [a, b]; st.country = null; st.shown = 100; document.getElementById("pick").value = ""; refresh(); }
const zoom = d3.zoom().scaleExtent([1, 12]).translateExtent([[0, 0], [W, H]])
  .on("zoom", ev => { k = ev.transform.k; g.attr("transform", ev.transform); hideTip(); if (S) drawMap(); });
svg.call(zoom).on("click", () => { if (st.country || st.link) select(null); });
document.getElementById("zin").onclick = () => svg.transition().duration(250).call(zoom.scaleBy, 1.8);
document.getElementById("zout").onclick = () => svg.transition().duration(250).call(zoom.scaleBy, 1 / 1.8);
document.getElementById("zreset").onclick = () => svg.transition().duration(300).call(zoom.transform, d3.zoomIdentity);

// ---------- topics by year heatmap ----------
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
function drawHeat() { if (st.hview === "bars") return drawTopicBars(); drawHeatmap(); }
const tcol = t => css("--t" + t);
// stacked bars: one bar per year, segments = topics; a selected topic moves to the baseline and the rest fade
function drawTopicBars() {
  const ps = P.filter(p => inPlace(p) && inAuthor(p));
  const cnt = TOPICS.map(() => YEARS.map(() => 0)), tot = YEARS.map(() => 0);
  ps.forEach(p => { cnt[p.t][p.y - 2000]++; tot[p.y - 2000]++; });
  const share = st.heat === "share", L = 44, Wd = 960, Hd = 300, top = 10, bottom = 44, bw = (Wd - L - 10) / YEARS.length;
  const ymax = share ? 1 : Math.max(1, d3.max(tot));
  const yS = d3.scaleLinear().domain([0, ymax]).nice().range([Hd - bottom, top]);
  const sv = d3.select("#heat").attr("viewBox", `0 0 ${Wd} ${Hd}`); sv.selectAll("*").remove();
  yS.ticks(4).forEach(v => { sv.append("line").attr("class", "grid").attr("x1", L).attr("x2", Wd - 10).attr("y1", yS(v)).attr("y2", yS(v));
    sv.append("text").attr("class", "ax").attr("x", L - 6).attr("y", yS(v) + 4).attr("text-anchor", "end").text(share ? Math.round(v * 100) + "%" : v); });
  const inRange = y => (st.phase < 0 || phaseOf(y) === st.phase) && (st.yearExact != null ? y === st.yearExact : y <= st.yearMax);
  const order = st.topic >= 0 ? [st.topic, ...TOPICS.map((_, i) => i).filter(i => i !== st.topic)] : TOPICS.map((_, i) => i);
  YEARS.forEach((y, j) => {
    const x = L + j * bw + 2; let acc = 0;
    if (tot[j]) order.forEach(t => {
      const v = share ? cnt[t][j] / tot[j] : cnt[t][j]; if (!v) return;
      const y0 = yS(acc), y1 = yS(acc + v); acc += v;
      const op = (st.topic >= 0 && t !== st.topic ? 0.22 : 1) * (inRange(y) ? 1 : 0.35);
      sv.append("rect").attr("class", "cell").attr("x", x).attr("width", bw - 4).attr("y", y1 + 1).attr("height", Math.max(0.5, y0 - y1 - 2)).attr("rx", 1.5)
        .attr("fill", tcol(t)).attr("opacity", op)
        .on("mousemove", ev => showTip(`<b>${TOPICS[t]}</b><br>${y}: ${pl(cnt[t][j], "paper")} of ${tot[j]} (${Math.round(100 * cnt[t][j] / tot[j])}%)<br><span class="m">Click to ${st.topic === t ? "list these papers" : "highlight this topic"}</span>`, ev))
        .on("mouseleave", hideTip)
        .on("click", () => { if (st.topic === t) { st.yearExact = y; st.shown = 100; syncControls(); showTab("papers"); } else setTopic(t); });
    });
    if (y % 2 === 0) sv.append("text").attr("class", "ax").attr("x", x + (bw - 4) / 2).attr("y", Hd - bottom + 16).attr("text-anchor", "middle").text(y);
    sv.append("text").attr("class", "ax").attr("x", x + (bw - 4) / 2).attr("y", Hd - 8).attr("text-anchor", "middle").attr("font-size", 10).text(tot[j] || "");
  });
  sv.append("text").attr("class", "ax").attr("x", L - 6).attr("y", Hd - 8).attr("text-anchor", "end").attr("font-size", 10).text("n =");
  const who = st.link ? `${byId[st.link[0]].name} and ${byId[st.link[1]].name} joint papers` : st.country ? `papers with an author in ${byId[st.country].name}` : st.author ? `papers by ${st.author.name}` : "all 812 papers";
  document.getElementById("heatSub").textContent = `${share ? "How each year's papers split across topics" : "Papers per year, by topic"}, for ${who}. ${st.topic >= 0 ? `${TOPICS[st.topic]} sits on the baseline; click one of its segments to list those papers.` : "Click a topic in the key or a segment to highlight it."} Faded years are outside the selected years. n = papers that year.`;
  const key = document.getElementById("heatKey");
  key.innerHTML = TOPICS.map((n, i) => `<button type="button" class="tk${st.topic === i ? " on" : ""}${st.topic >= 0 && st.topic !== i ? " off" : ""}" data-t="${i}"><span class="sq" style="background:${tcol(i)}"></span>${n}</button>`).join("");
  key.querySelectorAll("button").forEach(b => b.onclick = () => setTopic(+b.dataset.t === st.topic ? -1 : +b.dataset.t));
}
function drawHeatmap() {
  const ps = P.filter(p => inPlace(p) && inAuthor(p));
  const cnt = TOPICS.map(() => YEARS.map(() => 0)), tot = YEARS.map(() => 0);
  ps.forEach(p => { cnt[p.t][p.y - 2000]++; tot[p.y - 2000]++; });
  const share = st.heat === "share";
  const val = (t, j) => share ? (tot[j] ? cnt[t][j] / tot[j] : null) : cnt[t][j];
  const vmax = share ? Math.max(0.0001, d3.max(TOPICS.flatMap((_, t) => YEARS.map((_, j) => val(t, j) || 0)))) : Math.max(1, d3.max(cnt.flat()));
  const col = d3.scaleSequential(d3.interpolateLab(css("--heat-0"), css("--heat-1"))).domain([0, vmax]);
  const L = 330, cw = 26, ch = 26, top = 8, Wd = L + cw * YEARS.length + 8, Hd = top + ch * TOPICS.length + 44;
  const sv = d3.select("#heat").attr("viewBox", `0 0 ${Wd} ${Hd}`);
  sv.selectAll("*").remove();
  const inRange = y => (st.phase < 0 || phaseOf(y) === st.phase) && (st.yearExact != null ? y === st.yearExact : y <= st.yearMax);
  TOPICS.forEach((name, t) => {
    sv.append("text").attr("class", "rowlbl" + (t === st.topic ? " on" : "")).attr("x", L - 10).attr("y", top + t * ch + ch / 2 + 4).attr("text-anchor", "end")
      .text(name.length > 48 ? name.slice(0, 46) + "…" : name).on("click", () => setTopic(st.topic === t ? -1 : t))
      .append("title").text(`${name}: click to filter`);
    YEARS.forEach((y, j) => {
      const v = val(t, j);
      sv.append("rect").attr("class", "cell").attr("x", L + j * cw + 1).attr("y", top + t * ch + 1).attr("width", cw - 2).attr("height", ch - 2).attr("rx", 3)
        .attr("fill", v == null ? css("--chip") : col(v)).attr("opacity", inRange(y) ? 1 : 0.3)
        .attr("stroke", t === st.topic ? css("--hi") : "none").attr("stroke-width", t === st.topic ? 1.5 : 0)
        .on("mousemove", ev => showTip(`<b>${TOPICS[t]}</b><br>${y}: ${pl(cnt[t][j], "paper")}${tot[j] ? ` of ${tot[j]} (${Math.round(100 * cnt[t][j] / tot[j])}%)` : ""}<br><span class="m">Click to list these papers</span>`, ev))
        .on("mouseleave", hideTip)
        .on("click", () => { if (!cnt[t][j]) return; st.topic = t; st.yearExact = y; st.shown = 100; syncControls(); showTab("papers"); });
    });
  });
  YEARS.forEach((y, j) => { if (y % 2 === 0) sv.append("text").attr("class", "ax").attr("x", L + j * cw + cw / 2).attr("y", top + TOPICS.length * ch + 16).attr("text-anchor", "middle").text(y); });
  sv.append("text").attr("class", "ax").attr("x", L - 10).attr("y", Hd - 6).attr("text-anchor", "end").text("Papers that year");
  YEARS.forEach((y, j) => sv.append("text").attr("class", "ax").attr("x", L + j * cw + cw / 2).attr("y", Hd - 6).attr("text-anchor", "middle").attr("font-size", 10).text(tot[j] || ""));
  const who = st.link ? `${byId[st.link[0]].name} and ${byId[st.link[1]].name} joint papers` : st.country ? `papers with an author in ${byId[st.country].name}` : "all 812 papers";
  document.getElementById("heatSub").textContent = `${share ? "Each column shows how that year's papers split across topics" : "Number of papers per topic and year"}, for ${who}. Faded columns are outside the selected years. Click a topic to filter, or a cell to list its papers.`;
  const stops = d3.range(0, 1.0001, 0.25).map(f => f * vmax);
  document.getElementById("heatKey").innerHTML = `<span>${share ? "Share of year" : "Papers"}:</span>` + stops.map(v =>
    `<span><span class="sq" style="background:${col(v)}"></span>${share ? Math.round(v * 100) + "%" : Math.round(v)}</span>`).join("");
}

// ---------- region by year (paper-level) ----------
function drawReg() {
  const L = 40, Wd = 960, Hd = 230, top = 10, bottom = 30, bw = (Wd - L - 10) / YEARS.length;
  const rows = YEARS.map(y => {
    const frac = Array(REGIONS.length).fill(0); let n = 0;
    P.forEach(p => { if (p.y === y && inTopic(p) && inAuthor(p) && p.pr.length) { n++; p.pr.forEach(r => frac[r] += 1 / p.pr.length); } });
    return { y, n, frac };
  });
  const sv = d3.select("#reg").attr("viewBox", `0 0 ${Wd} ${Hd}`); sv.selectAll("*").remove();
  const yS = d3.scaleLinear().domain([0, 1]).range([Hd - bottom, top]);
  [0, 0.5, 1].forEach(v => { sv.append("line").attr("class", "grid").attr("x1", L).attr("x2", Wd - 10).attr("y1", yS(v)).attr("y2", yS(v));
    sv.append("text").attr("class", "ax").attr("x", L - 6).attr("y", yS(v) + 4).attr("text-anchor", "end").text(v * 100 + "%"); });
  const inRange = y => (st.phase < 0 || phaseOf(y) === st.phase) && (st.yearExact != null ? y === st.yearExact : y <= st.yearMax);
  rows.forEach((r, j) => {
    let acc = 0; const x = L + j * bw + 2;
    if (r.n) REGIONS.forEach((reg, i) => {
      const f = r.frac[i] / r.n; if (!f) return;
      const y0 = yS(acc), y1 = yS(acc + f); acc += f;
      sv.append("rect").attr("x", x).attr("width", bw - 4).attr("y", y1 + 1).attr("height", Math.max(0, y0 - y1 - 2)).attr("rx", 1.5)
        .attr("fill", REGION[reg]).attr("opacity", inRange(r.y) ? 1 : 0.3)
        .on("mousemove", ev => showTip(`<b>${r.y}</b> · ${pl(r.n, "paper")} with affiliations<br>${REGIONS.map((q, k2) => r.frac[k2] ? `${q}: ${Math.round(100 * r.frac[k2] / r.n)}%` : "").filter(Boolean).join("<br>")}`, ev))
        .on("mouseleave", hideTip);
    });
    if (r.y % 2 === 0) sv.append("text").attr("class", "ax").attr("x", x + (bw - 4) / 2).attr("y", Hd - 10).attr("text-anchor", "middle").text(r.y);
  });
  document.getElementById("regSub").textContent = `Share of authorship by region among papers with affiliations${st.topic >= 0 ? `, ${TOPICS[st.topic]} only` : ""}. A paper with authors in several regions is split evenly between them. Faded bars are outside the selected years.`;
}

// ---------- topics by region (paper-level, fractional) ----------
function drawTR() {
  const val = TOPICS.map(() => REGIONS.map(() => 0)), rt = REGIONS.map(() => 0);
  P.forEach(p => { if (!inTime(p) || !inAuthor(p) || !p.pr.length) return; p.pr.forEach(r => { val[p.t][r] += 1 / p.pr.length; rt[r] += 1 / p.pr.length; }); });
  const share = (t, r) => rt[r] ? val[t][r] / rt[r] : null;
  const vmax = Math.max(0.0001, d3.max(TOPICS.flatMap((_, t) => REGIONS.map((_, r) => share(t, r) || 0))));
  const col = d3.scaleSequential(d3.interpolateLab(css("--heat-0"), css("--heat-1"))).domain([0, vmax]);
  const L = 330, cw = 96, ch = 28, top = 34, Wd = L + cw * REGIONS.length + 8, Hd = top + ch * TOPICS.length + 26;
  const sv = d3.select("#tr").attr("viewBox", `0 0 ${Wd} ${Hd}`); sv.selectAll("*").remove();
  REGIONS.forEach((r, j) => {
    sv.append("rect").attr("x", L + j * cw + 1).attr("y", 24).attr("width", cw - 2).attr("height", 4).attr("rx", 2).attr("fill", REGION[r]);
    sv.append("text").attr("class", "ax").attr("x", L + j * cw + cw / 2).attr("y", 16).attr("text-anchor", "middle").text(r);
    sv.append("text").attr("class", "ax").attr("x", L + j * cw + cw / 2).attr("y", Hd - 6).attr("text-anchor", "middle").text(rt[j] ? `${Math.round(rt[j])} papers` : "none");
  });
  TOPICS.forEach((name, t) => {
    sv.append("text").attr("class", "rowlbl" + (t === st.topic ? " on" : "")).attr("x", L - 10).attr("y", top + t * ch + ch / 2 + 4).attr("text-anchor", "end")
      .text(name.length > 48 ? name.slice(0, 46) + "…" : name).on("click", () => setTopic(st.topic === t ? -1 : t));
    REGIONS.forEach((r, j) => {
      const v = share(t, j);
      sv.append("rect").attr("class", "cell").attr("x", L + j * cw + 1).attr("y", top + t * ch + 1).attr("width", cw - 2).attr("height", ch - 2).attr("rx", 3)
        .attr("fill", v == null ? css("--chip") : col(v)).attr("stroke", t === st.topic ? css("--hi") : "none").attr("stroke-width", 1.5)
        .on("mousemove", ev => showTip(`<b>${r}</b> · ${TOPICS[t]}<br>${v == null ? "No papers" : `${Math.round(100 * v)}% of ${r} authorship (${val[t][j].toFixed(1)} of ${Math.round(rt[j])} papers)`}`, ev))
        .on("mouseleave", hideTip).on("click", () => setTopic(st.topic === t ? -1 : t));
      if (v != null && v >= 0.005) sv.append("text").attr("x", L + j * cw + cw / 2).attr("y", top + t * ch + ch / 2 + 4).attr("text-anchor", "middle")
        .attr("font-size", 11).attr("pointer-events", "none").attr("fill", v > vmax * 0.55 ? css("--surface") : css("--fg")).style("font-variant-numeric", "tabular-nums").text(Math.round(100 * v) + "%");
    });
  });
  document.getElementById("trSub").textContent = `Each column shows how a region's authorship splits across topics, for ${timeLabel()}. Read down a column to see what each region focuses on. Paper-level affiliations; a paper with authors in several regions is split evenly. Click a topic to filter.`;
  const stops = d3.range(0, 1.0001, 0.25).map(f => f * vmax);
  document.getElementById("trKey").innerHTML = `<span>Share of region:</span>` + stops.map(v => `<span><span class="sq" style="background:${col(v)}"></span>${Math.round(v * 100)}%</span>`).join("");
}

// ---------- papers table ----------
function papersInView() {
  const q = st.q.toLowerCase();
  return P.filter(p => inTime(p) && inTopic(p) && inPlace(p) && inAuthor(p) && (!q || p.title.toLowerCase().includes(q) || p.authors.toLowerCase().includes(q)))
    .sort((a, b) => b.y - a.y || a.title.localeCompare(b.title));
}
function drawPapers(list) {
  const rows = list.slice(0, st.shown);
  const noLoc = list.filter(p => !p.au.length).length;
  document.getElementById("pinfo").textContent = `${pl(list.length, "paper")}${list.length > rows.length ? `, showing ${rows.length}` : ""}${noLoc ? ` · ${noLoc} without author locations (not on the map)` : ""}`;
  document.getElementById("ptable").innerHTML = rows.length ? rows.map(p => {
    const ac = p.ac.map(c => NAMES[c]), pc = p.pc.map(c => NAMES[c]);
    const where = ac.length ? ac.join(", ") : pc.length ? `<span class="pl">${pc.join(", ")}</span>` : `<span class="pl">No location found</span>`;
    const title = p.doi ? `<a href="${p.doi.startsWith("http") ? encodeURI(p.doi) : "https://doi.org/" + encodeURI(p.doi)}" target="_blank" rel="noopener">${esc(p.title)}</a>`
      : `<a href="https://scholar.google.com/scholar?q=${encodeURIComponent('"' + p.title + '"')}" target="_blank" rel="noopener" title="No DOI: opens a Google Scholar search for this title">${esc(p.title)}</a> <span class="pl">(no DOI, Scholar search)</span>`;
    return `<tr><td class="y">${p.y}</td><td class="t">${title}<div class="au">${esc(p.authors)}</div></td><td class="c">${where}</td><td class="tp">${TOPICS[p.t]}</td></tr>`;
  }).join("") : `<tr><td colspan="4" class="empty">No papers match these filters.</td></tr>`;
  const more = document.getElementById("pmore");
  more.innerHTML = list.length > rows.length ? `<button type="button" class="btn" id="moreBtn">Show ${Math.min(100, list.length - rows.length)} more</button>` : "";
  const mb = document.getElementById("moreBtn"); if (mb) mb.onclick = () => { st.shown += 100; drawPapers(list); };
}

// ---------- save as PNG ----------
// Renders the chart in the light theme at 2.5x resolution, with title, active filters, legend and source line.
// In the Claude viewer, files go through the downloads capability; a saved copy of this page in a browser has no window.claude
const DL = (window.claude && window.claude.use ? window.claude.use("downloads") : Promise.resolve(null)).catch(() => null);
const SAVE_PROPS = ["fill", "fill-opacity", "stroke", "stroke-width", "stroke-opacity", "opacity", "font-family", "font-size", "font-weight",
  "font-style", "paint-order", "stroke-linejoin", "stroke-linecap", "text-anchor", "dominant-baseline", "vector-effect", "visibility"];
function filterLine() {
  const parts = [timeLabel(), st.topic >= 0 ? TOPICS[st.topic] : "All topics"];
  if (st.country) parts.push(byId[st.country].name);
  if (st.link) parts.push(`${byId[st.link[0]].name} and ${byId[st.link[1]].name}`);
  if (st.author) parts.push("Author: " + st.author.name);
  return parts.join(" · ");
}
function toast(msg) { const el = document.getElementById("toast"); el.textContent = msg; el.hidden = false; clearTimeout(toast.t); toast.t = setTimeout(() => el.hidden = true, 3500); }
function wrapText(ctx, text, maxW) {
  const words = text.split(" "), lines = []; let line = "";
  words.forEach(w => { const tr = line ? line + " " + w : w; if (ctx.measureText(tr).width > maxW && line) { lines.push(line); line = w; } else line = tr; });
  if (line) lines.push(line); return lines;
}
// Europe inset for the saved map: its own projection, links with both ends in Europe, same circle and line scales
const EU_BOX = [[-11, 34.5], [32, 66.5]];
function euBoxOnMain() {
  const pts = []; for (let i = 0; i <= 10; i++) { const t = i / 10;
    pts.push(proj([EU_BOX[0][0] + t * (EU_BOX[1][0] - EU_BOX[0][0]), EU_BOX[0][1]]), proj([EU_BOX[0][0] + t * (EU_BOX[1][0] - EU_BOX[0][0]), EU_BOX[1][1]]));
    pts.push(proj([EU_BOX[0][0], EU_BOX[0][1] + t * (EU_BOX[1][1] - EU_BOX[0][1])]), proj([EU_BOX[1][0], EU_BOX[0][1] + t * (EU_BOX[1][1] - EU_BOX[0][1])])); }
  const [x0, x1] = d3.extent(pts, p => p[0]), [y0, y1] = d3.extent(pts, p => p[1]); return [x0, y0, x1 - x0, y1 - y0];
}
function europeInsetSVG(IW, IH) {
  const pe = d3.geoMercator().fitExtent([[6, 6], [IW - 6, IH - 6]], { type: "MultiPoint", coordinates: [EU_BOX[0], EU_BOX[1], [EU_BOX[0][0], EU_BOX[1][1]], [EU_BOX[1][0], EU_BOX[0][1]]] })
    .clipExtent([[0, 0], [IW, IH]]);
  const path = d3.geoPath(pe), cs = sel => getComputedStyle(document.querySelector(sel));
  const land = cs("#map path.land"), arcS = css("--link"), hi = css("--hi"), ring = css("--ring"), fg = css("--fg"), bg = css("--surface"), muted = css("--muted");
  const eu = nodes.filter(n => n.region === "Europe" && S.node[n.id].papers > 0), xy = {}; eu.forEach(n => xy[n.id] = pe([n.lon, n.lat]));
  const focus = st.link ? new Set(st.link) : st.country ? new Set([st.country, ...partners(st.country).map(p => p.id)]) : null;
  const hiE = e => isLink(e) || (st.country && (e.s === st.country || e.t === st.country));
  const arcs = S.cross.filter(e => xy[e.s] && xy[e.t]).sort((a, b) => a.w - b.w).map(e => {
    const [ax, ay] = xy[e.s], [bx, by] = xy[e.t], dx = bx - ax, dy = by - ay;
    const op = focus ? (hiE(e) ? 0.95 : 0.07) : 0.3 + 0.55 * Math.min(1, e.w / WMAX_FIXED);
    return `<path d="M${ax},${ay}Q${(ax + bx) / 2 - dy * 0.18},${(ay + by) / 2 + dx * 0.18} ${bx},${by}" fill="none" stroke="${hiE(e) ? hi : arcS}" stroke-linecap="round" stroke-width="${0.8 + 4.2 * Math.min(1, e.w / WMAX_FIXED)}" stroke-opacity="${op}"/>`; }).join("");
  const circ = eu.slice().sort((a, b) => S.node[b.id].papers - S.node[a.id].papers).map(n => `<circle cx="${xy[n.id][0]}" cy="${xy[n.id][1]}" r="${rad(n.id) * 0.8}" fill="${REGION[n.region]}" fill-opacity="${focus ? (focus.has(n.id) ? 1 : 0.18) : 1}" stroke="${ring}" stroke-width="1.2"/>`).join("");
  const order = eu.filter(n => !focus || focus.has(n.id)).sort((a, b) => S.node[b.id].papers - S.node[a.id].papers);
  const lbl = placeLabels(order.map(n => ({ x: xy[n.id][0], y: xy[n.id][1], r: rad(n.id) * 0.8, text: n.name, fs: 8.5 })), IW, IH)
    .map(d => `<text x="${d.x}" y="${d.y}" text-anchor="${d.anchor}" font-family="Inter, system-ui, sans-serif" font-size="8.5" font-weight="600" fill="${fg}" stroke="${bg}" stroke-width="2.5" stroke-linejoin="round" paint-order="stroke">${esc(d.it.text)}</text>`).join("");
  const landP = DATA.land.features.map(f => `<path d="${path(f) || ""}" fill="${land.fill}" stroke="${land.stroke}" stroke-width="0.5"/>`).join("");
  return `<svg xmlns="http://www.w3.org/2000/svg" width="${IW}" height="${IH}" viewBox="0 0 ${IW} ${IH}"><rect width="${IW}" height="${IH}" fill="${bg}"/>${landP}${arcs}${circ}${lbl}<rect x="0.5" y="0.5" width="${IW - 1}" height="${IH - 1}" fill="none" stroke="${muted}" stroke-width="1"/></svg>`;
}
async function svgImage(str) {
  const url = URL.createObjectURL(new Blob([str], { type: "image/svg+xml;charset=utf-8" }));
  try { return await new Promise((res, rej) => { const im = new Image(); im.onload = () => res(im); im.onerror = rej; im.src = url; }); } finally { setTimeout(() => URL.revokeObjectURL(url), 1000); }
}
async function savePNG(kind) {
  const cfg = {
    map: { svg: "#map", title: "Who writes with whom at dg.o", legend: () => REGIONS.map(r => ({ c: REGION[r], l: r, round: true })),
           note: "Circle size = papers with an author in that country. Line width = papers co-written across two countries. Same-country collaboration not drawn." },
    heat: { svg: "#heat", title: "Topics by year", legend: () => st.hview === "bars" ? TOPICS.map((n, i) => ({ c: css("--t" + i), l: n })) : heatKeyItems("#heatKey"),
            note: st.hview === "bars" ? (st.heat === "share" ? "Share of each year's papers by topic." : "Papers per year by topic.") : "Heatmap of topics by year." },
    reg: { svg: "#reg", title: "Where authors were based, by year", legend: () => REGIONS.map(r => ({ c: REGION[r], l: r })),
           note: "Share of authorship by region among papers with affiliations; multi-region papers split evenly." },
    tr: { svg: "#tr", title: "Topics by region", legend: () => heatKeyItems("#trKey"), note: "Share of each region's authorship by topic; paper-level affiliations." },
  }[kind];
  const root = document.documentElement, prevTheme = root.getAttribute("data-theme");
  root.setAttribute("data-theme", "light"); hideTip();
  const INSET = kind === "map" && k === 1; EXPORT_LBL = INSET;
  if (kind === "heat") drawHeat(); if (kind === "reg") drawReg(); if (kind === "tr") drawTR(); if (kind === "map") drawMap();
  try {
    const src = document.querySelector(cfg.svg), vb = src.viewBox.baseVal, clone = src.cloneNode(true);
    const a = [src, ...src.querySelectorAll("*")], b = [clone, ...clone.querySelectorAll("*")];
    a.forEach((el, i) => { const cs = getComputedStyle(el); b[i].setAttribute("style", SAVE_PROPS.map(p => `${p}:${cs.getPropertyValue(p)}`).join(";")); });
    clone.setAttribute("xmlns", "http://www.w3.org/2000/svg"); clone.setAttribute("width", vb.width); clone.setAttribute("height", vb.height);
    if (INSET) { const [bx, by, bw, bh] = euBoxOnMain(), r = document.createElementNS("http://www.w3.org/2000/svg", "rect");
      Object.entries({ x: bx, y: by, width: bw, height: bh, fill: "none", stroke: css("--muted"), "stroke-width": 0.8, "stroke-dasharray": "4 3" }).forEach(([a, v]) => r.setAttribute(a, v));
      clone.querySelector("g").appendChild(r); }
    const legend = cfg.legend(), bg = css("--surface"), fg = css("--fg"), muted = css("--muted");
    const IW = 340, IH = 340, insW = INSET ? IW + 18 : 0;
    const SC = 2.5, pad = 24, Wc = vb.width + pad * 2 + insW, font = '"Inter", system-ui, -apple-system, "Segoe UI", sans-serif';
    const meas = document.createElement("canvas").getContext("2d");
    meas.font = `12px ${font}`; const sub = wrapText(meas, filterLine(), Wc - pad * 2), noteL = wrapText(meas, cfg.note, Wc - pad * 2);
    let lx = 0, rows = 1; legend.forEach(it => { const w = 18 + meas.measureText(it.l).width + 16; if (lx + w > Wc - pad * 2) { rows++; lx = 0; } lx += w; });
    const keyH = kind === "map" ? 2 * rScale()(PMAX) + 34 : 0;
    const headH = 24 + 26 + sub.length * 16 + 10, legH = rows * 20 + 8 + keyH, footH = noteL.length * 16 + 22, Hc = headH + vb.height + legH + footH + pad;
    const cv = document.createElement("canvas"); cv.width = Math.round(Wc * SC); cv.height = Math.round(Hc * SC);
    const ctx = cv.getContext("2d"); ctx.scale(SC, SC);
    ctx.fillStyle = bg; ctx.fillRect(0, 0, Wc, Hc);
    ctx.fillStyle = fg; ctx.font = `600 20px ${font}`; ctx.fillText(cfg.title, pad, 24 + 18);
    ctx.fillStyle = muted; ctx.font = `12px ${font}`; sub.forEach((l, i) => ctx.fillText(l, pad, 24 + 26 + 12 + i * 16));
    const url = URL.createObjectURL(new Blob([new XMLSerializer().serializeToString(clone)], { type: "image/svg+xml;charset=utf-8" }));
    const img = await new Promise((res, rej) => { const im = new Image(); im.onload = () => res(im); im.onerror = rej; im.src = url; });
    ctx.drawImage(img, pad, headH, vb.width, vb.height); URL.revokeObjectURL(url);
    if (INSET) {
      const ix = pad + vb.width + 18, iy = headH + Math.max(18, (vb.height - IH) / 2);
      ctx.fillStyle = muted; ctx.font = `12px ${font}`; ctx.fillText("Europe (links within Europe)", ix, iy - 6);
      ctx.drawImage(await svgImage(europeInsetSVG(IW, IH)), ix, iy, IW, IH);
    }
    let x = pad, y = headH + vb.height + 14; ctx.font = `12px ${font}`;
    legend.forEach(it => {
      const w = 18 + ctx.measureText(it.l).width + 16; if (x + w > Wc - pad) { x = pad; y += 20; }
      ctx.fillStyle = it.c; ctx.beginPath(); if (it.round) ctx.arc(x + 5, y - 4, 5, 0, Math.PI * 2); else ctx.rect(x, y - 9, 10, 10); ctx.fill();
      ctx.fillStyle = fg; ctx.fillText(it.l, x + 16, y); x += w;
    });
    if (kind === "map") {   // line-width and circle-size keys, same scales as the map
      const rMax = rScale()(PMAX), base = y + 22 + 2 * rMax; let kx = pad;
      ctx.fillStyle = fg; ctx.font = `600 12px ${font}`; ctx.fillText("Link between countries", kx, y + 22); ctx.font = `12px ${font}`;
      [1, 6, 12].forEach((w, i) => { const ly = y + 40 + i * 16; ctx.strokeStyle = css("--link"); ctx.globalAlpha = 0.3 + 0.55 * Math.min(1, w / WMAX_FIXED);
        ctx.lineWidth = 0.8 + 4.2 * Math.min(1, w / WMAX_FIXED); ctx.lineCap = "round"; ctx.beginPath(); ctx.moveTo(kx, ly - 4); ctx.lineTo(kx + 34, ly - 4); ctx.stroke();
        ctx.globalAlpha = 1; ctx.fillStyle = fg; ctx.fillText(pl(w, "paper"), kx + 44, ly); });
      kx += 190; ctx.font = `600 12px ${font}`; ctx.fillText("Country size", kx, y + 22); ctx.font = `12px ${font}`;
      let lastY = y + 22;
      [PMAX, 50, 10].forEach(v => { const r = rScale()(v), cy = base - r + 6, top = cy - r; ctx.strokeStyle = muted; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.arc(kx + rMax, cy, r, 0, Math.PI * 2); ctx.stroke();
        const ty = Math.max(top + 4, lastY + 15); lastY = ty;
        ctx.beginPath(); ctx.moveTo(kx + rMax, top); ctx.lineTo(kx + 2 * rMax + 8, ty - 4); ctx.stroke();
        ctx.fillStyle = fg; ctx.fillText(pl(v, "paper"), kx + 2 * rMax + 12, ty); });
      y += keyH;
    }
    ctx.fillStyle = muted; y += 24;
    noteL.forEach((l, i) => ctx.fillText(l, pad, y + i * 16));
    ctx.font = `11px ${font}`; ctx.fillText("Source: DGRL v2.1 dg.o corpus (812 papers); affiliations from OpenAlex, Crossref, ROR and manual lookups; k=10 BERTopic topics.", pad, y + noteL.length * 16 + 2);
    const name = ("dgo_" + kind + "_" + filterLine()).toLowerCase().replace(/[^a-z0-9]+/g, "_").replace(/^_|_$/g, "").slice(0, 90) + ".png";
    const blob = await new Promise(res => cv.toBlob(res, "image/png"));
    const dl = await DL;
    if (dl) {   // inside the Claude viewer: the viewer confirms the save
      try { await dl.save({ filename: name, data: blob }); toast("Saved " + name); }
      catch (e) { toast(e && e.code === "declined" ? "Image not saved." : e && e.code === "rate_limited" ? "A save is already waiting for your answer." : "Saving is not available here. Download the HTML file and try there."); }
    } else {    // the downloaded HTML file in a normal browser
      const link = document.createElement("a"); link.href = URL.createObjectURL(blob); link.download = name;
      document.body.appendChild(link); link.click(); link.remove(); setTimeout(() => URL.revokeObjectURL(link.href), 4000);
      toast("Saved " + name);
    }
  } catch (e) { toast("Could not create the image in this view. Open the downloaded HTML file in a browser and try again."); }
  finally {
    EXPORT_LBL = false;
    if (prevTheme == null) root.removeAttribute("data-theme"); else root.setAttribute("data-theme", prevTheme);
    if (kind === "heat") drawHeat(); if (kind === "reg") drawReg(); if (kind === "tr") drawTR(); if (kind === "map") drawMap();
  }
}
function heatKeyItems(sel) {
  return [...document.querySelectorAll(sel + " > span")].map(s => { const sq = s.querySelector(".sq"); return sq ? { c: sq.style.background, l: s.textContent.trim() } : null; }).filter(Boolean);
}
document.querySelectorAll("button.save").forEach(b => b.onclick = () => savePNG(b.dataset.save));

// ---------- controls ----------
// the period buttons are cumulative: each sets the "through" year (phase filtering is kept in code but unused)
function setPhase(p) { stopPlay(); st.phase = p; st.yearExact = null; syncControls(); refresh(); }
function setTopic(t) { st.topic = t; st.shown = 100; syncControls(); refresh(); }
function setYear(y) { st.yearMax = y; st.yearExact = null; syncControls(); refresh(); }
function syncControls() {
  document.querySelectorAll("#phase button").forEach(b => b.setAttribute("aria-pressed", st.phase < 0 && st.yearExact == null && +b.dataset.y === st.yearMax));
  document.getElementById("topic").value = String(st.topic);
  document.getElementById("yr").value = st.yearMax; document.getElementById("yrOut").textContent = st.yearExact != null ? "Year " + st.yearExact : st.yearMax;
  document.getElementById("yr").disabled = st.yearExact != null;
}
document.querySelectorAll("#phase button").forEach(b => b.onclick = () => { stopPlay(); st.phase = -1; setYear(+b.dataset.y); });
const tsel = document.getElementById("topic");
TOPICS.forEach((t, i) => tsel.add(new Option(`${t} (${P.filter(p => p.t === i).length})`, String(i))));
tsel.onchange = () => setTopic(+tsel.value);
const pick = document.getElementById("pick");
nodes.slice().sort((a, b) => a.name.localeCompare(b.name)).forEach(n => pick.add(new Option(n.name, n.id)));
pick.onchange = () => select(pick.value || null);
// author search: names map to every author ID with that name (duplicates kept separate in the data)
const byName = new Map(); AUTHORS.forEach((n, i) => { if (!byName.has(n)) byName.set(n, []); byName.get(n).push(i); });
document.getElementById("authorList").innerHTML = [...byName.keys()].sort((a, b) => a.localeCompare(b)).map(n => `<option value="${esc(n)}"></option>`).join("");
function setAuthor(name) {
  st.author = name && byName.has(name) ? { name, ids: byName.get(name) } : null;
  if (!st.author) document.getElementById("author").value = "";
  st.country = null; st.link = null; st.shown = 100; document.getElementById("pick").value = ""; refresh();
  if (st.author && st.tab === "map") { const ids = nodes.filter(n => S.node[n.id].papers > 0).map(n => n.id); if (ids.length) flyTo(ids); }
}
document.getElementById("author").addEventListener("change", e => setAuthor(e.target.value.trim()));
document.getElementById("author").addEventListener("input", e => { const v = e.target.value.trim(); if (byName.has(v)) setAuthor(v); else if (!v && st.author) setAuthor(null); });
document.getElementById("yr").oninput = e => { stopPlay(); setYear(+e.target.value); };
function stopPlay() { if (playing) { clearInterval(playing); playing = null; document.getElementById("play").textContent = "▶ Play"; } }
document.getElementById("play").onclick = () => {
  if (playing) return stopPlay();
  st.phase = -1; st.yearExact = null; st.yearMax = st.yearMax >= 2024 ? 2000 : st.yearMax; syncControls(); refresh();
  document.getElementById("play").textContent = "❚❚ Pause";
  playing = setInterval(() => { if (st.yearMax >= 2024) return stopPlay(); st.yearMax++; syncControls(); refresh(); }, 700);
};
let qT; document.getElementById("q").oninput = e => { clearTimeout(qT); qT = setTimeout(() => { st.q = e.target.value.trim(); st.shown = 100; refresh(); }, 150); };
document.getElementById("heatMode").querySelectorAll("button").forEach(b => b.onclick = () => {
  st.heat = b.dataset.m; document.querySelectorAll("#heatMode button").forEach(x => x.setAttribute("aria-pressed", x === b)); drawHeat(); });
document.getElementById("heatView").querySelectorAll("button").forEach(b => b.onclick = () => {
  st.hview = b.dataset.v; document.querySelectorAll("#heatView button").forEach(x => x.setAttribute("aria-pressed", x === b)); drawHeat(); });

// ---------- tabs (hash routing) ----------
// tabs switch in the page itself; the address hash is only a bookmark when the viewer allows it
function showTab(t) {
  st.tab = t;
  try { if (location.hash.slice(1) !== t) history.replaceState(null, "", "#" + t); } catch (e) {}
  ["map", "topics", "papers"].forEach(x => { document.getElementById("view-" + x).hidden = x !== t; document.getElementById("tab-" + x).setAttribute("aria-selected", x === t); });
  hideTip(); refresh();
}
document.querySelectorAll(".tabs button").forEach(b => b.onclick = () => showTab(b.dataset.tab));
addEventListener("hashchange", () => { const h = location.hash.slice(1); if (["map", "topics", "papers"].includes(h) && h !== st.tab) showTab(h); });
matchMedia("(prefers-color-scheme: dark)").addEventListener("change", () => { if (st.tab === "topics") drawHeat(); });

function refresh() {
  S = compute();
  strip(); chips();
  const list = papersInView();
  document.getElementById("pcount").textContent = fmt(list.length);
  if (st.tab === "map") { drawMap(); panel(list.length); }
  if (st.tab === "topics") { drawHeat(); drawReg(); drawTR(); }
  if (st.tab === "papers") drawPapers(list);
}
syncControls(); showTab(["map", "topics", "papers"].includes(location.hash.slice(1)) ? location.hash.slice(1) : "map");
</script>
'''
html_out = os.path.join(OUT, 'country_collaboration_map.html')
with open(html_out, 'w', encoding='utf-8') as fh:
    fh.write('<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">'
             '<style>body{margin:0}[hidden]{display:none!important}</style></head><body>\n'
             + TEMPLATE.replace('__N_AUTH__', str(A[A.country.astype(str).str.strip() != ''].row_id.nunique())).replace('__N_PAPER__', str((m.countries_all.astype(str).str.strip() != '').sum())).replace('__DATA__', json.dumps(DATA, separators=(',', ':'))) + '\n</body></html>')
print('saved', html_out, f'({os.path.getsize(html_out) / 1024:.0f} KB). Download it from Drive and open it in any browser.')


## Run manifest
Records the run date, package versions, checksums and row counts for every input and output, and the headline numbers, in `run_manifest.json`.

In [ ]:
import importlib.metadata as im, networkx as nx
def md5_of(p):
    h = hashlib.md5()
    with open(p, 'rb') as fh:
        for ch in iter(lambda: fh.read(1 << 20), b''): h.update(ch)
    return h.hexdigest()
def describe(p):
    d = {'path': p.replace(REV + '/', ''), 'exists': os.path.exists(p)}
    if d['exists']:
        d['md5'] = md5_of(p); d['bytes'] = os.path.getsize(p)
        if p.endswith('.csv'): d['rows'] = len(pd.read_csv(p))
    return d
INPUTS = [os.path.join(IN, f) for f in ['dgrl_topics_k10.csv', 'dgrl_topics_k15.csv', 'dgrl_embeddings.npy']] + \
         [os.path.join(IN, f) for f in ['openalex_matches_cache.json', 'openalex_authorships.csv', 'manual_lookups_log.csv', 'affiliation_lookups_log.csv', 'web_affiliation_lookups_2026-11-19.csv']]
OUTPUTS = [os.path.join(REV, f) for f in ['affiliations.csv', 'abstracts_backfill.csv', 'still_missing_abstracts.csv', 'still_missing_for_manual_entry.csv',
           'dgrl_master.csv', 'key_numbers_by_phase.csv', 'authorships_stage1.csv', 'authorships_raw.csv', 'authorships.csv',
           'dblp_author_matches.csv', 'author_location_recovery_log.csv', 'affiliation_lookups_combined.csv', 'network_missing_papers.csv', 'dblp/dblp_dgo_records.json', 'api_cache.json']] + \
          [os.path.join(OUT, f) for f in ['coauthorship_network.gexf', 'coauthorship_nodes.csv', 'coauthorship_edges.csv', 'duplicate_author_decisions.csv',
           'country_network.gexf', 'country_nodes.csv', 'country_edges.csv', 'institution_network.gexf', 'institution_nodes.csv', 'institution_edges.csv',
           'country_collaboration_map.png', 'country_collaboration_map.html',
           'country_collaboration_map_2000_2005.png', 'country_collaboration_map_2000_2012.png', 'country_collaboration_rankings.csv', 'country_pair_rankings.csv']] + [os.path.join(REV, 'geo', 'ne_110m_admin_0_countries.geojson')]
save_api_cache()
master = pd.read_csv(os.path.join(REV, 'dgrl_master.csv'))
Af = pd.read_csv(os.path.join(REV, 'authorships.csv'))
Gc = nx.read_gexf(os.path.join(OUT, 'coauthorship_network.gexf'))
headline = {
    'papers': len(master),
    'papers_with_abstract': int((master.abstract.fillna('') != '').sum()),
    'papers_with_affiliation_country': int((master.n_countries.fillna(0) > 0).sum()),
    'papers_in_coauthor_network': int(Af.row_id.nunique()),
    'papers_with_author_level_location': int(Af.dropna(subset=['country']).row_id.nunique()),
    'authors': Gc.number_of_nodes(), 'coauthor_links': Gc.number_of_edges(),
    'largest_connected_group': len(max(nx.connected_components(Gc), key=len)),
    'papers_moved_topic_k10': int(master.get('topic_k10_moved', pd.Series(dtype=bool)).fillna(False).sum()),
}
manifest = {
    'run_started': RUN_STARTED, 'run_finished': datetime.datetime.now().isoformat(timespec='seconds'), 'refresh_api': REFRESH_API,
    'api_cache_responses': len(API),
    'packages': {p: im.version(p) for p in ['pandas', 'numpy', 'rapidfuzz', 'networkx', 'lxml', 'sentence-transformers', 'pycountry-convert', 'requests']},
    'manual_decisions': {'false_match_titles': FALSE_MATCH_TITLES, 'title_fix': {str(k): v for k, v in TITLE_FIX.items()},
                         'manual_same_pairs': MANUAL_SAME_PAIRS, 'phases': PHASES},
    'inputs': [describe(p) for p in INPUTS], 'outputs': [describe(p) for p in OUTPUTS], 'headline': headline,
}
json.dump(manifest, open(os.path.join(REV, 'run_manifest.json'), 'w'), indent=1)
print(json.dumps(headline, indent=1))
missing = [d['path'] for d in manifest['inputs'] + manifest['outputs'] if not d['exists']]
print("\nMissing files:", missing or 'none')
print("Saved run_manifest.json")